# Master’s Thesis — Final Analysis Notebook

## Purpose

This notebook starts from the saved analysis checkpoints and contains only the final analyses, tables, figures, and robustness checks used in the thesis.

The complete data-processing and model-development history is preserved in:

`01_thesis_full_analysis_ARCHIVE.ipynb`

## Primary analysis decisions

- Primary analytical panel: 430 diary entries from 40 participants
- Primary LLM measure: mean of Azure prompts 1–3
- Prompt 4 is reported separately
- Primary network moderator: number of nominated friends
- Language indicators are the model outcomes
- Mood is decomposed into within-person and between-person components
- Word count, gender, and socioeconomic status are included as controls

In [ ]:
# ============================================================
# STARTUP CELL — run once after opening or restarting notebook
# ============================================================

from pathlib import Path
import json
import os
import glob
import warnings

import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from IPython.display import display

warnings.filterwarnings("default")

ROOT = Path.cwd() / "Data Sharing"
OUTPUTS = ROOT / "outputs"
FIGURES = ROOT / "figures_final"
TABLES = ROOT / "tables_final"

OUTPUTS.mkdir(exist_ok=True)
FIGURES.mkdir(exist_ok=True)
TABLES.mkdir(exist_ok=True)

print("Notebook directory:", Path.cwd())
print("Data root:", ROOT)
print("Data root exists:", ROOT.exists())

In [ ]:
# ============================================================
# CHECK REQUIRED CHECKPOINT FILES
# ============================================================

required_files = {
    "analysis panel": ROOT / "analysis_ready_checkpoint.parquet",
    "scored survey": ROOT / "df_survey_scored_FINAL.parquet",
    "person-level data": ROOT / "person_FINAL.parquet",
    "original LLM ratings": ROOT / "llm_ratings_azure.parquet",
    "LLM v2 ratings": ROOT / "llm_v2_ratings_azure.parquet",
    "human-LLM validation": OUTPUTS / "validation_all106_HUMAN_LLM_JOINED.csv",
}

file_status = []

for label, path in required_files.items():
    file_status.append({
        "file": label,
        "exists": path.exists(),
        "path": str(path)
    })

file_status = pd.DataFrame(file_status)
display(file_status)

missing_files = file_status.loc[~file_status["exists"], "file"].tolist()

if missing_files:
    print("\nMissing files:", missing_files)
else:
    print("\nAll required checkpoint files are available.")

In [ ]:
# ============================================================
# LOAD FINAL WORKING DATA
# ============================================================

panel = pd.read_parquet(
    ROOT / "analysis_ready_checkpoint.parquet"
)

survey = pd.read_parquet(
    ROOT / "df_survey_scored_FINAL.parquet"
)

person = pd.read_parquet(
    ROOT / "person_FINAL.parquet"
)

llm_original = pd.read_parquet(
    ROOT / "llm_ratings_azure.parquet"
)

print("Analysis panel:", panel.shape)
print("Panel participants:", panel["participantid"].nunique())
print("Scored survey:", survey.shape)
print("Person-level data:", person.shape)
print("Original LLM ratings:", llm_original.shape)

In [ ]:
# ============================================================
# INTEGRITY CHECKS
# ============================================================

expected_columns = [
    "participantid",
    "social_conn_llm",
    "sent_score",
    "social_density",
    "neg_emotion_density",
    "mood_NA_within_z",
    "mood_NA_between_z",
    "mood_PA_within_z",
    "mood_PA_between_z",
    "PHQ9_total_z",
    "GAD7_total_z",
    "n_friends",
    "n_friends_z",
    "gender_female",
    "family_ses_rev_z",
    "nwords_z",
]

missing_columns = [
    col for col in expected_columns
    if col not in panel.columns
]

print("Panel rows:", len(panel))
print("Participants:", panel["participantid"].nunique())
print("Duplicate dataframe rows:", panel.duplicated().sum())
print("Missing required columns:", missing_columns)

assert len(panel) == 430, (
    f"Expected 430 diary rows, found {len(panel)}."
)

assert panel["participantid"].nunique() == 40, (
    "Expected 40 participants in the analytical panel."
)

assert not missing_columns, (
    f"Required columns are missing: {missing_columns}"
)

print("\nStartup integrity checks passed.")

In [ ]:
# ============================================================
# CREATE WORKING COPY
# ============================================================

analysis = panel.copy()

print("Original checkpoint:", panel.shape)
print("Working copy:", analysis.shape)
print("Same object in memory:", analysis is panel)


# 1. Final Data Audit

This section verifies the final analytical dataset before any models or thesis tables are rerun.

## Restart Instructions

After restarting VS Code or the computer, run only these cells in order:

1. Startup cell
2. Check required checkpoint files
3. Load final working data
4. Integrity checks
5. Create working copy

Do not rerun the archived notebook or any Azure, DeepL, transcript-processing, or sentiment-generation cells unless the underlying data construction must be changed.

In [ ]:
# ============================================================
# FINAL DATA AUDIT 1 — MISSING VALUES IN FINAL VARIABLES
# ============================================================

audit_variables = [
    # Language outcomes
    "social_conn_llm",
    "sent_score",
    "social_density",
    "neg_emotion_density",

    # Mood predictors
    "mood_NA_within_z",
    "mood_NA_between_z",
    "mood_PA_within_z",
    "mood_PA_between_z",

    # Clinical predictors
    "PHQ9_total_z",
    "GAD7_total_z",

    # Network moderator
    "n_friends",
    "n_friends_z",

    # Controls
    "gender_female",
    "family_ses_rev_z",
    "nwords_z",
]

missing_audit = pd.DataFrame({
    "variable": audit_variables,
    "available_rows": [
        analysis[v].notna().sum()
        for v in audit_variables
    ],
    "missing_rows": [
        analysis[v].isna().sum()
        for v in audit_variables
    ],
})

missing_audit["percent_missing"] = (
    missing_audit["missing_rows"] / len(analysis) * 100
).round(2)

display(missing_audit)

print("\nTotal analytical rows:", len(analysis))

In [ ]:
# ============================================================
# FINAL DATA AUDIT 2 — IDENTIFY THE MISSING DENSITY ROW
# ============================================================

density_missing = analysis[
    analysis["social_density"].isna()
    | analysis["neg_emotion_density"].isna()
].copy()

safe_columns = [
    "participantid",
    "date",
    "slot",
    "n_words_clean",
    "nwords_z",
    "social_conn_llm",
    "sent_score",
    "social_density",
    "neg_emotion_density",
]

safe_columns = [
    col for col in safe_columns
    if col in density_missing.columns
]

print("Rows with at least one missing density outcome:", len(density_missing))
display(density_missing[safe_columns])

In [ ]:
# ============================================================
# FINAL DATA AUDIT 3 — VERIFY WHY DENSITIES ARE MISSING
# ============================================================

import re

row = density_missing.iloc[0]
text = row["text_clean"]

whitespace_tokens = str(text).split()

german_word_tokens = re.findall(
    r"\b[a-zäöüß]+\b",
    str(text).lower()
)

general_word_tokens = re.findall(
    r"\b\w+\b",
    str(text).lower()
)

print("Stored n_words_clean:", row["n_words_clean"])
print("Whitespace token count:", len(whitespace_tokens))
print("General word-token count:", len(general_word_tokens))
print("German alphabetic-token count:", len(german_word_tokens))
print("Character count:", len(str(text)))
print("Contains alphabetic characters:", bool(re.search(r"[A-Za-zÄÖÜäöüß]", str(text))))

In [ ]:
# ============================================================
# FINAL DATA AUDIT 4 — COMMON SAMPLE FOR COMPARATIVE MODELS
# ============================================================

language_outcomes = [
    "social_conn_llm",
    "sent_score",
    "social_density",
    "neg_emotion_density",
]

analysis["common_language_sample"] = (
    analysis[language_outcomes].notna().all(axis=1)
)

analysis_common = analysis[
    analysis["common_language_sample"]
].copy()

print("Full analytical panel:", analysis.shape)
print("Common comparison sample:", analysis_common.shape)
print(
    "Participants in common sample:",
    analysis_common["participantid"].nunique()
)
print(
    "Rows excluded:",
    (~analysis["common_language_sample"]).sum()
)

print("\nOutcome availability in common sample:")
print(
    analysis_common[language_outcomes]
    .notna()
    .sum()
)

In [ ]:
# ============================================================
# FINAL DATA AUDIT 5 — SAVE AUDITED WORKING CHECKPOINT
# ============================================================

audit_checkpoint = ROOT / "analysis_ready_audited_v1.parquet"

analysis.to_parquet(
    audit_checkpoint,
    index=False
)

print("Saved:", audit_checkpoint)
print("Saved rows:", len(analysis))
print(
    "Common-sample rows:",
    int(analysis["common_language_sample"].sum())
)
print(
    "Excluded from common sample:",
    int((~analysis["common_language_sample"]).sum())
)

## Audit Decision 1 — Common Analytical Sample

The full analytical panel contains 430 diary entries from 40 participants. One diary entry contained no alphabetic word tokens after text cleaning. Consequently, social-word density and negative-emotion-word density were undefined for that entry, although the LLM and sentiment measures were available.

For all regression tables comparing the four language-based outcomes side by side, a common complete-case sample of 429 diary entries from 40 participants will be used. The full 430-entry panel is retained for diagnostics and analyses that do not require direct comparison across all four language outcomes.

The excluded row was not assigned a density value of zero because zero would indicate that valid words were present but none matched the relevant lexicon. In this case, no valid word-token denominator existed.

In [ ]:
# ============================================================
# FINAL DATA AUDIT 6 — NETWORK VARIABLES AT PARTICIPANT LEVEL
# ============================================================

network_variables = [
    col for col in ["n_friends", "n_contacts", "network_size"]
    if col in analysis.columns
]

print("Network variables available:", network_variables)

# Check whether each network variable is constant within participants
consistency_rows = []

for variable in network_variables:
    unique_within_person = (
        analysis.groupby("participantid")[variable]
        .nunique(dropna=False)
    )

    consistency_rows.append({
        "variable": variable,
        "participants": len(unique_within_person),
        "participants_with_more_than_one_value":
            int((unique_within_person > 1).sum()),
        "maximum_values_within_one_participant":
            int(unique_within_person.max()),
    })

network_consistency = pd.DataFrame(consistency_rows)

print("\nWithin-participant consistency:")
display(network_consistency)

# Create one row per participant for network descriptives
network_person = (
    analysis.groupby("participantid")[network_variables]
    .first()
    .reset_index()
)

network_descriptives = (
    network_person[network_variables]
    .agg(["count", "mean", "std", "min", "median", "max"])
    .T
    .reset_index()
    .rename(columns={"index": "variable"})
    .round(3)
)

print("\nParticipant-level network descriptives:")
display(network_descriptives)

# Correlations between available network variables
if len(network_variables) > 1:
    print("\nParticipant-level correlations:")
    display(
        network_person[network_variables]
        .corr()
        .round(3)
    )

In [ ]:
# ============================================================
# FINAL DATA AUDIT 7 — VERIFY STANDARDIZED FRIEND COUNT
# ============================================================

friends_person = (
    analysis[
        ["participantid", "n_friends", "n_friends_z"]
    ]
    .drop_duplicates("participantid")
    .copy()
)

friends_person["n_friends_z_recalculated"] = (
    friends_person["n_friends"]
    - friends_person["n_friends"].mean()
) / friends_person["n_friends"].std(ddof=1)

friends_person["absolute_difference"] = (
    friends_person["n_friends_z"]
    - friends_person["n_friends_z_recalculated"]
).abs()

print("Participants checked:", len(friends_person))

print(
    "Raw friend count: mean =",
    round(friends_person["n_friends"].mean(), 3),
    "| SD =",
    round(friends_person["n_friends"].std(ddof=1), 3),
    "| range =",
    (
        friends_person["n_friends"].min(),
        friends_person["n_friends"].max(),
    ),
)

print(
    "Stored n_friends_z: mean =",
    round(friends_person["n_friends_z"].mean(), 6),
    "| SD =",
    round(friends_person["n_friends_z"].std(ddof=1), 6),
)

print(
    "Maximum difference between stored and recalculated z-score:",
    friends_person["absolute_difference"].max(),
)

print(
    "Stored standardization matches participant-level recalculation:",
    np.allclose(
        friends_person["n_friends_z"],
        friends_person["n_friends_z_recalculated"],
        equal_nan=True,
    ),
)


In [ ]:
# ============================================================
# FINAL DATA AUDIT 8 — TRACE HOW n_friends_z WAS STANDARDIZED
# ============================================================

# A. Recalculate using all 430 diary rows
analysis["n_friends_z_rowweighted_check"] = (
    analysis["n_friends"]
    - analysis["n_friends"].mean()
) / analysis["n_friends"].std(ddof=1)

# B. Recalculate using one row per participant
friends_person = (
    analysis[
        ["participantid", "n_friends", "n_friends_z"]
    ]
    .drop_duplicates("participantid")
    .copy()
)

participant_mean = friends_person["n_friends"].mean()
participant_sd = friends_person["n_friends"].std(ddof=1)

analysis["n_friends_z_participant_check"] = (
    analysis["n_friends"] - participant_mean
) / participant_sd

# C. Compare the stored version with both possibilities
rowweighted_difference = (
    analysis["n_friends_z"]
    - analysis["n_friends_z_rowweighted_check"]
).abs()

participant_difference = (
    analysis["n_friends_z"]
    - analysis["n_friends_z_participant_check"]
).abs()

print("Diary-row-weighted raw mean:", round(analysis["n_friends"].mean(), 6))
print("Diary-row-weighted raw SD:", round(analysis["n_friends"].std(ddof=1), 6))

print("\nParticipant-level raw mean:", round(participant_mean, 6))
print("Participant-level raw SD:", round(participant_sd, 6))

print("\nMaximum difference: stored vs diary-row-weighted:")
print(rowweighted_difference.max())

print("\nMaximum difference: stored vs participant-level:")
print(participant_difference.max())

print("\nStored z-score matches diary-row-weighted calculation:")
print(
    np.allclose(
        analysis["n_friends_z"],
        analysis["n_friends_z_rowweighted_check"],
        equal_nan=True,
    )
)

print("\nStored z-score matches participant-level calculation:")
print(
    np.allclose(
        analysis["n_friends_z"],
        analysis["n_friends_z_participant_check"],
        equal_nan=True,
    )
)

In [ ]:
# ============================================================
# FINAL DATA AUDIT 9 — PARTICIPANT-STANDARDIZED FRIEND COUNT
# ============================================================

# One stable friend-count value per participant
friends_reference = (
    analysis[
        ["participantid", "n_friends"]
    ]
    .drop_duplicates("participantid")
    .copy()
)

# Participant-level reference values
friends_mean_person = friends_reference["n_friends"].mean()
friends_sd_person = friends_reference["n_friends"].std(ddof=1)

# Create a new variable; preserve the previous stored version
analysis["n_friends_z_person"] = (
    analysis["n_friends"] - friends_mean_person
) / friends_sd_person

# Check once per participant
friends_check = (
    analysis[
        [
            "participantid",
            "n_friends",
            "n_friends_z",
            "n_friends_z_person",
        ]
    ]
    .drop_duplicates("participantid")
)

print("Participants:", len(friends_check))
print("Participant-level raw mean:", round(friends_mean_person, 6))
print("Participant-level raw SD:", round(friends_sd_person, 6))

print("\nNew participant-standardized variable:")
print(
    "Mean =",
    round(friends_check["n_friends_z_person"].mean(), 6)
)
print(
    "SD =",
    round(friends_check["n_friends_z_person"].std(ddof=1), 6)
)
print(
    "Minimum =",
    round(friends_check["n_friends_z_person"].min(), 3)
)
print(
    "Maximum =",
    round(friends_check["n_friends_z_person"].max(), 3)
)

print("\nOld variable preserved:")
print("n_friends_z" in analysis.columns)

print("New variable created:")
print("n_friends_z_person" in analysis.columns)

In [ ]:
# ============================================================
# FINAL DATA AUDIT 10 — SAVE NETWORK-AUDITED CHECKPOINT
# ============================================================

temporary_network_columns = [
    "n_friends_z_rowweighted_check",
    "n_friends_z_participant_check",
]

analysis = analysis.drop(
    columns=[
        col for col in temporary_network_columns
        if col in analysis.columns
    ]
)

network_audit_checkpoint = (
    ROOT / "analysis_ready_audited_v2.parquet"
)

analysis.to_parquet(
    network_audit_checkpoint,
    index=False
)

print("Saved:", network_audit_checkpoint)
print("Rows:", len(analysis))
print("Participants:", analysis["participantid"].nunique())
print(
    "Common comparison rows:",
    int(analysis["common_language_sample"].sum())
)
print(
    "Participant-standardized friends variable available:",
    "n_friends_z_person" in analysis.columns
)
print(
    "Temporary diagnostic columns remaining:",
    [
        col for col in temporary_network_columns
        if col in analysis.columns
    ]
)

## Audit Decision 2 — Friendship Network Moderator

The primary network moderator is the number of nominated friends (`n_friends`). This measure is constant within every participant and corresponds to the friendship-based theoretical argument underlying the moderation analysis.

The existing standardized variable (`n_friends_z`) had been calculated across repeated diary rows. Consequently, participants who contributed more diary entries had greater influence on the mean and standard deviation used for standardization.

Because friendship-network size is a person-level characteristic, a new variable (`n_friends_z_person`) was calculated using one observation per participant. Across the 40 participants, the raw number of nominated friends had a mean of 2.80, a standard deviation of 2.82, and a range from 0 to 18. The participant-standardized variable therefore has a participant-level mean of 0 and standard deviation of 1.

The original `n_friends_z` variable is retained for reproducibility of earlier results. The corrected `n_friends_z_person` variable will be used in the final Family 2 moderation models. The combined `network_size` variable will not be used because combining friends and contacts may double-count the same nominated individuals and cannot be interpreted as total network size without alter-level de-duplication.

In [ ]:
# ============================================================
# FINAL DATA AUDIT 11 — VERIFY PERSON-LEVEL VARIABLES
# ============================================================

person_level_variables = [
    "PHQ9_total",
    "GAD7_total",
    "gender_female",
    "family_ses_rev",
    "n_friends",
]

# Keep only variables available in the audited dataset
person_level_variables = [
    variable
    for variable in person_level_variables
    if variable in analysis.columns
]

print("Person-level variables checked:")
print(person_level_variables)

consistency_results = []

for variable in person_level_variables:

    values_per_participant = (
        analysis.groupby("participantid")[variable]
        .nunique(dropna=False)
    )

    consistency_results.append({
        "variable": variable,
        "participants_checked": len(values_per_participant),
        "participants_with_more_than_one_value":
            int((values_per_participant > 1).sum()),
        "maximum_values_for_one_participant":
            int(values_per_participant.max()),
        "missing_diary_rows":
            int(analysis[variable].isna().sum()),
    })

person_variable_audit = pd.DataFrame(consistency_results)

display(person_variable_audit)

In [ ]:
# ============================================================
# FINAL DATA AUDIT 12 — CREATE PARTICIPANT-LEVEL ANALYTICAL DATA
# ============================================================

participant_analysis = (
    analysis[
        ["participantid"] + person_level_variables
    ]
    .drop_duplicates("participantid")
    .copy()
)

print("Participant-level dataset shape:", participant_analysis.shape)
print(
    "Unique participants:",
    participant_analysis["participantid"].nunique()
)
print(
    "Duplicate participant IDs:",
    participant_analysis["participantid"].duplicated().sum()
)

print("\nMissing values:")
print(
    participant_analysis[
        person_level_variables
    ].isna().sum()
)

In [ ]:
# ============================================================
# FINAL DATA AUDIT 13 — PARTICIPANT-LEVEL DESCRIPTIVES
# ============================================================

continuous_person_variables = [
    "PHQ9_total",
    "GAD7_total",
    "family_ses_rev",
    "n_friends",
]

participant_descriptives = (
    participant_analysis[continuous_person_variables]
    .agg(["count", "mean", "std", "min", "median", "max"])
    .T
    .reset_index()
    .rename(columns={
        "index": "variable",
        "count": "N",
        "mean": "Mean",
        "std": "SD",
        "min": "Minimum",
        "median": "Median",
        "max": "Maximum",
    })
)

participant_descriptives[
    ["Mean", "SD", "Minimum", "Median", "Maximum"]
] = participant_descriptives[
    ["Mean", "SD", "Minimum", "Median", "Maximum"]
].round(2)

participant_descriptives["N"] = (
    participant_descriptives["N"].astype(int)
)

print("Participant-level continuous descriptives:")
display(participant_descriptives)

gender_counts = (
    participant_analysis["gender_female"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("gender_female")
    .reset_index(name="N")
)

gender_counts["Percent"] = (
    gender_counts["N"] / len(participant_analysis) * 100
).round(1)

print("\nParticipant-level gender distribution:")
display(gender_counts)

In [ ]:
# ============================================================
# FINAL DATA AUDIT 14 — VERIFY GENDER AND SES CODING
# ============================================================

possible_columns = [
    "gender",
    "gender_female",
    "family_ses",
    "family_ses_num",
    "family_ses_rev",
]

available_columns = [
    col for col in possible_columns
    if col in analysis.columns
]

print("Available coding columns:")
print(available_columns)

coding_check = (
    analysis[
        ["participantid"] + available_columns
    ]
    .drop_duplicates("participantid")
    .copy()
)

if "gender" in coding_check.columns:
    print("\nRaw gender categories:")
    print(
        coding_check["gender"]
        .value_counts(dropna=False)
    )

if {"gender", "gender_female"}.issubset(coding_check.columns):
    print("\nRaw gender × gender_female:")
    display(
        pd.crosstab(
            coding_check["gender"],
            coding_check["gender_female"],
            dropna=False,
        )
    )

if "family_ses" in coding_check.columns:
    print("\nRaw family SES categories:")
    print(
        coding_check["family_ses"]
        .value_counts(dropna=False)
        .sort_index()
    )

ses_display_columns = [
    col for col in [
        "family_ses",
        "family_ses_num",
        "family_ses_rev",
    ]
    if col in coding_check.columns
]

if ses_display_columns:
    print("\nSES coding combinations:")
    display(
        coding_check[ses_display_columns]
        .drop_duplicates()
        .sort_values(ses_display_columns[-1])
    )

In [ ]:
# ============================================================
# FINAL DATA AUDIT 15 — ADD RAW GENDER AND SES LABELS
# ============================================================

raw_person_variables = (
    analysis[
        [
            "participantid",
            "gender",
            "family_ses",
            "family_ses_num",
        ]
    ]
    .drop_duplicates("participantid")
    .copy()
)

participant_analysis = participant_analysis.merge(
    raw_person_variables,
    on="participantid",
    how="left",
    validate="one_to_one",
)

print("Updated participant-level dataset:", participant_analysis.shape)
print("Participants:", participant_analysis["participantid"].nunique())
print("Duplicate participant IDs:", participant_analysis["participantid"].duplicated().sum())

print("\nRaw gender distribution:")
gender_final = (
    participant_analysis["gender"]
    .value_counts(dropna=False)
    .rename_axis("Gender")
    .reset_index(name="N")
)

gender_final["Percent"] = (
    gender_final["N"] / len(participant_analysis) * 100
).round(1)

display(gender_final)

print("\nSES direction check:")
print("Minimum reversed SES:", participant_analysis["family_ses_rev"].min())
print("Maximum reversed SES:", participant_analysis["family_ses_rev"].max())
print("Higher family_ses_rev means greater socioeconomic advantage.")

In [ ]:
# ============================================================
# FINAL DATA AUDIT 16 — VERIFY DIARY-LEVEL VARIABLES
# ============================================================

candidate_diary_variables = [
    # Language outcomes
    "social_conn_llm",
    "sent_score",
    "social_density",
    "neg_emotion_density",

    # Raw time-varying mood variables
    "mood_PA",
    "mood_NA",

    # Text length
    "n_words_clean",
]

diary_variables = [
    variable
    for variable in candidate_diary_variables
    if variable in analysis.columns
]

missing_candidate_variables = [
    variable
    for variable in candidate_diary_variables
    if variable not in analysis.columns
]

print("Diary-level variables available:")
print(diary_variables)

print("\nCandidate variables missing:")
print(missing_candidate_variables)

diary_variable_check = pd.DataFrame({
    "variable": diary_variables,
    "available_rows": [
        analysis[variable].notna().sum()
        for variable in diary_variables
    ],
    "missing_rows": [
        analysis[variable].isna().sum()
        for variable in diary_variables
    ],
    "unique_values": [
        analysis[variable].nunique(dropna=True)
        for variable in diary_variables
    ],
})

display(diary_variable_check)

In [ ]:
# ============================================================
# FINAL DATA AUDIT 17 — DIARY-LEVEL DESCRIPTIVES
# ============================================================

diary_descriptives = (
    analysis[diary_variables]
    .agg([
        "count",
        "mean",
        "std",
        "min",
        "median",
        "max",
    ])
    .T
    .reset_index()
    .rename(columns={
        "index": "variable",
        "count": "N",
        "mean": "Mean",
        "std": "SD",
        "min": "Minimum",
        "median": "Median",
        "max": "Maximum",
    })
)

diary_descriptives["N"] = (
    diary_descriptives["N"].astype(int)
)

diary_descriptives[
    ["Mean", "SD", "Minimum", "Median", "Maximum"]
] = diary_descriptives[
    ["Mean", "SD", "Minimum", "Median", "Maximum"]
].round(3)

print("Diary-entry-level descriptives:")
display(diary_descriptives)

In [ ]:
# ============================================================
# FINAL DATA AUDIT 18 — SAVE CORRECTED DESCRIPTIVE TABLES
# ============================================================

# Add readable labels
diary_labels = {
    "social_conn_llm": "LLM social connectedness",
    "sent_score": "Sentiment score",
    "social_density": "Social-word density",
    "neg_emotion_density": "Negative-emotion density",
    "mood_PA": "Positive mood",
    "mood_NA": "Negative mood",
    "n_words_clean": "Diary word count",
}

participant_labels = {
    "PHQ9_total": "PHQ-9 depression",
    "GAD7_total": "GAD-7 anxiety",
    "family_ses_rev": "Family socioeconomic position",
    "n_friends": "Number of nominated friends",
}

diary_descriptives_final = diary_descriptives.copy()
diary_descriptives_final["variable"] = (
    diary_descriptives_final["variable"].map(diary_labels)
)

participant_descriptives_final = participant_descriptives.copy()
participant_descriptives_final["variable"] = (
    participant_descriptives_final["variable"].map(participant_labels)
)

# Save tables
diary_path = TABLES / "Table_1a_diary_level_descriptives.csv"
participant_path = TABLES / "Table_1b_participant_level_descriptives.csv"
gender_path = TABLES / "Table_1c_gender_distribution.csv"

diary_descriptives_final.to_csv(diary_path, index=False)
participant_descriptives_final.to_csv(participant_path, index=False)
gender_final.to_csv(gender_path, index=False)

print("Saved:")
print(diary_path)
print(participant_path)
print(gender_path)

print("\nDiary-level table:")
display(diary_descriptives_final)

print("\nParticipant-level table:")
display(participant_descriptives_final)

print("\nGender distribution:")
display(gender_final)

## Audit Decision 3 — Diary-Level and Participant-Level Descriptives

Descriptive statistics are separated according to the analytical level of each variable.

Diary-level descriptives are calculated across diary entries for the four language-based outcomes, raw positive mood, raw negative mood, and diary word count. The full available sample is reported for each variable: 430 entries for the LLM measure, sentiment, mood, and word count, and 429 entries for the two lexical-density outcomes.

Participant-level descriptives are calculated using one row per participant for PHQ-9 depression, GAD-7 anxiety, family socioeconomic position, friendship-network size, and gender. These statistics describe the 40 participants included in the analytical diary panel and avoid giving greater descriptive weight to participants who completed more diary entries.

Family socioeconomic position is reported on the reversed raw 1–10 scale, with higher values indicating greater socioeconomic advantage. Gender is reported using the original categories: female, male, and diverse. In the regression models, the control variable `gender_female` is coded 1 for female and 0 for non-female participants.

Raw positive and negative mood scores are used in the descriptive table for interpretability. The standardized within-person and between-person mood components are used in the regression models and are documented in the analysis-strategy section rather than treated as the primary mood descriptives.

In [ ]:
# ============================================================
# FINAL DATA AUDIT 19 — PREPARE EXPANDED CORRELATION SAMPLE
# ============================================================

# Recreate the common 429-row sample from the saved flag
correlation_sample = analysis[
    analysis["common_language_sample"]
].copy()

correlation_variables = [
    # Language outcomes
    "social_conn_llm",
    "sent_score",
    "social_density",
    "neg_emotion_density",

    # Within- and between-person mood components
    "mood_NA_within_z",
    "mood_NA_between_z",
    "mood_PA_within_z",
    "mood_PA_between_z",

    # Person-level mental-health predictors
    "PHQ9_total_z",
    "GAD7_total_z",

    # Corrected participant-standardized moderator
    "n_friends_z_person",

    # Text-length control
    "nwords_z",
]

missing_correlation_variables = [
    variable
    for variable in correlation_variables
    if variable not in correlation_sample.columns
]

correlation_missingness = pd.DataFrame({
    "variable": correlation_variables,
    "available_rows": [
        correlation_sample[v].notna().sum()
        for v in correlation_variables
    ],
    "missing_rows": [
        correlation_sample[v].isna().sum()
        for v in correlation_variables
    ],
})

print("Correlation sample rows:", len(correlation_sample))
print(
    "Participants:",
    correlation_sample["participantid"].nunique()
)
print(
    "Missing required correlation variables:",
    missing_correlation_variables
)

display(correlation_missingness)


In [ ]:
# ============================================================
# FINAL DATA AUDIT 20 — EXPANDED CORRELATION MATRIX
# ============================================================

correlation_labels = {
    "social_conn_llm": "1. LLM social connectedness",
    "sent_score": "2. Sentiment score",
    "social_density": "3. Social-word density",
    "neg_emotion_density": "4. Negative-emotion density",
    "mood_NA_within_z": "5. Negative mood, within",
    "mood_NA_between_z": "6. Negative mood, between",
    "mood_PA_within_z": "7. Positive mood, within",
    "mood_PA_between_z": "8. Positive mood, between",
    "PHQ9_total_z": "9. PHQ-9 depression",
    "GAD7_total_z": "10. GAD-7 anxiety",
    "n_friends_z_person": "11. Number of nominated friends",
    "nwords_z": "12. Diary word count",
}

correlation_matrix = (
    correlation_sample[correlation_variables]
    .apply(pd.to_numeric, errors="coerce")
    .corr(method="pearson")
)

correlation_matrix_labeled = correlation_matrix.rename(
    index=correlation_labels,
    columns=correlation_labels,
)

print("Expanded Pearson correlation matrix:")
display(correlation_matrix_labeled.round(3))

In [ ]:
# ============================================================
# FINAL DATA AUDIT 21 — SAVE FINAL EXPANDED CORRELATION TABLE
# ============================================================

correlation_path = (
    TABLES / "Table_2_expanded_correlations_common429.csv"
)

correlation_matrix_labeled.round(3).to_csv(
    correlation_path
)

print("Saved:", correlation_path)
print("Sample: 429 diary entries from 40 participants")
print("Table 2a will not be retained as a separate main-text table.")

## Audit Decision 4 — Expanded Correlation Table

The final thesis will include one expanded Pearson correlation table rather than separate Tables 2a and 2b. The smaller language-indicator matrix is fully contained within the expanded table and is therefore retained only as a diagnostic output.

All correlations in the final expanded table are calculated using the common complete-case sample of 429 diary entries from 40 participants. This avoids the variable-specific sample sizes used in the earlier response memo.

The table includes the four language-based outcomes, within-person and between-person positive and negative mood, PHQ-9 depression, GAD-7 anxiety, participant-standardized number of nominated friends, and diary word count.

The correlations are descriptive because diary entries are nested within participants and person-level variables are repeated across entries. Conventional Pearson-correlation significance tests are therefore not used for substantive inference. Inferential conclusions are based on the mixed-effects models.

The smaller language-indicator correlation table will not appear separately in the main thesis.

In [ ]:
# ============================================================
# FINAL MODELS 1 — FIT FAMILY 1 ON COMMON 429-ROW SAMPLE
# ============================================================

import warnings
import statsmodels.formula.api as smf

family1_data = analysis[
    analysis["common_language_sample"]
].copy()

family1_outcomes = [
    "social_conn_llm",
    "sent_score",
    "social_density",
    "neg_emotion_density",
]

family1_predictors = [
    "mood_NA_within_z",
    "mood_NA_between_z",
    "mood_PA_within_z",
    "mood_PA_between_z",
    "gender_female",
    "family_ses_rev_z",
    "nwords_z",
]

# Confirm one complete common sample for every model
required_family1 = (
    ["participantid"]
    + family1_outcomes
    + family1_predictors
)

print("Family 1 rows:", len(family1_data))
print(
    "Family 1 participants:",
    family1_data["participantid"].nunique()
)
print(
    "Missing values across required variables:",
    int(family1_data[required_family1].isna().sum().sum())
)

family1_models = {}
family1_health = []

optimizers = ["lbfgs", "bfgs", "cg", "powell", "nm"]

for outcome in family1_outcomes:

    formula = (
        f"{outcome} ~ "
        "mood_NA_within_z + mood_NA_between_z + "
        "mood_PA_within_z + mood_PA_between_z + "
        "gender_female + family_ses_rev_z + nwords_z"
    )

    fitted_model = None
    optimizer_used = None
    warning_messages = []
    last_error = None

    for optimizer in optimizers:

        try:
            with warnings.catch_warnings(record=True) as caught:
                warnings.simplefilter("always")

                candidate = smf.mixedlm(
                    formula=formula,
                    data=family1_data,
                    groups=family1_data["participantid"],
                ).fit(
                    reml=True,
                    method=optimizer,
                    maxiter=2000,
                    disp=False,
                )

            warning_messages.extend(
                [str(w.message) for w in caught]
            )

            if candidate.converged:
                fitted_model = candidate
                optimizer_used = optimizer
                break

        except Exception as error:
            last_error = str(error)

    if fitted_model is not None:

        random_intercept_variance = float(
            fitted_model.cov_re.iloc[0, 0]
        )

        residual_variance = float(
            fitted_model.scale
        )

        family1_models[outcome] = fitted_model

        family1_health.append({
            "outcome": outcome,
            "N_observations": int(fitted_model.nobs),
            "N_participants":
                int(family1_data["participantid"].nunique()),
            "optimizer": optimizer_used,
            "converged": fitted_model.converged,
            "random_intercept_variance":
                random_intercept_variance,
            "residual_variance":
                residual_variance,
            "boundary_warning": any(
                "boundary" in message.lower()
                or "singular" in message.lower()
                for message in warning_messages
            ),
            "last_error": "",
        })

    else:

        family1_health.append({
            "outcome": outcome,
            "N_observations": len(family1_data),
            "N_participants":
                family1_data["participantid"].nunique(),
            "optimizer": None,
            "converged": False,
            "random_intercept_variance": np.nan,
            "residual_variance": np.nan,
            "boundary_warning": None,
            "last_error": last_error,
        })

family1_health = pd.DataFrame(family1_health)

print("\nFamily 1 model-health check:")
display(family1_health)

In [ ]:
# ============================================================
# FINAL MODELS 2 — INSPECT FAMILY 1 ESTIMATES
# ============================================================

family1_terms = [
    "mood_NA_within_z",
    "mood_NA_between_z",
    "mood_PA_within_z",
    "mood_PA_between_z",
    "gender_female",
    "family_ses_rev_z",
    "nwords_z",
]

family1_results_rows = []

for outcome, model in family1_models.items():

    var_participant = float(model.cov_re.iloc[0, 0])
    var_residual = float(model.scale)

    icc = (
        var_participant
        / (var_participant + var_residual)
    )

    for term in family1_terms:

        family1_results_rows.append({
            "outcome": outcome,
            "term": term,
            "beta": model.params[term],
            "SE": model.bse[term],
            "p": model.pvalues[term],
            "N": int(model.nobs),
            "participants": int(model.model.n_groups),
            "ICC": icc,
        })

family1_results = pd.DataFrame(
    family1_results_rows
)

family1_results_display = (
    family1_results.copy()
)

family1_results_display[
    ["beta", "SE", "p", "ICC"]
] = family1_results_display[
    ["beta", "SE", "p", "ICC"]
].round(4)

print("Family 1 coefficient results:")
display(family1_results_display)

In [ ]:
# ============================================================
# FINAL MODELS 3 — FAMILY 1 CLUSTER-ROBUST CHECK
# ============================================================

family1_cluster_models = {}
family1_cluster_rows = []

for outcome in family1_outcomes:

    formula = (
        f"{outcome} ~ "
        "mood_NA_within_z + mood_NA_between_z + "
        "mood_PA_within_z + mood_PA_between_z + "
        "gender_female + family_ses_rev_z + nwords_z"
    )

    cluster_model = smf.ols(
        formula=formula,
        data=family1_data,
    ).fit(
        cov_type="cluster",
        cov_kwds={
            "groups": family1_data["participantid"]
        },
    )

    family1_cluster_models[outcome] = cluster_model

    for term in [
        "mood_NA_within_z",
        "mood_NA_between_z",
        "mood_PA_within_z",
        "mood_PA_between_z",
    ]:

        mixed_model = family1_models[outcome]

        family1_cluster_rows.append({
            "outcome": outcome,
            "term": term,
            "mixed_beta": mixed_model.params[term],
            "mixed_SE": mixed_model.bse[term],
            "mixed_p": mixed_model.pvalues[term],
            "cluster_beta": cluster_model.params[term],
            "cluster_SE": cluster_model.bse[term],
            "cluster_p": cluster_model.pvalues[term],
        })

family1_robustness = pd.DataFrame(
    family1_cluster_rows
)

family1_robustness[
    [
        "mixed_beta",
        "mixed_SE",
        "mixed_p",
        "cluster_beta",
        "cluster_SE",
        "cluster_p",
    ]
] = family1_robustness[
    [
        "mixed_beta",
        "mixed_SE",
        "mixed_p",
        "cluster_beta",
        "cluster_SE",
        "cluster_p",
    ]
].round(4)

display(family1_robustness)

In [ ]:
# ============================================================
# FINAL MODELS 4 — SAVE FAMILY 1 ROBUSTNESS RESULTS
# ============================================================

family1_robustness_path = (
    TABLES / "Family1_mixed_vs_clustered_robustness.csv"
)

family1_robustness.to_csv(
    family1_robustness_path,
    index=False,
)

print("Saved:", family1_robustness_path)
print("Rows saved:", len(family1_robustness))
print("Common sample: 429 diary entries from 40 participants")

In [ ]:
# ============================================================
# FINAL MODELS 5 — SAVE FAMILY 1 MIXED-MODEL RESULTS
# ============================================================

family1_model_rows = []

for outcome, model in family1_models.items():

    participant_variance = float(
        model.cov_re.iloc[0, 0]
    )
    residual_variance = float(
        model.scale
    )
    icc = (
        participant_variance
        / (participant_variance + residual_variance)
    )

    for term in model.params.index:

        # Group Var is an internal variance parameter rather than
        # a fixed-effect coefficient for the thesis table.
        if term == "Group Var":
            continue

        family1_model_rows.append({
            "outcome": outcome,
            "term": term,
            "beta": float(model.params[term]),
            "SE": float(model.bse[term]),
            "p": float(model.pvalues[term]),
            "N_observations": int(model.nobs),
            "N_participants": int(model.model.n_groups),
            "participant_intercept_variance":
                participant_variance,
            "residual_variance":
                residual_variance,
            "ICC": icc,
        })

family1_saved_results = pd.DataFrame(
    family1_model_rows
)

family1_results_path = (
    TABLES / "Family1_mixed_models_common429.csv"
)

family1_saved_results.to_csv(
    family1_results_path,
    index=False,
)

print("Saved:", family1_results_path)
print("Rows saved:", len(family1_saved_results))
print("Outcomes:", family1_saved_results["outcome"].nunique())
print(
    "N observations by outcome:",
    family1_saved_results.groupby("outcome")[
        "N_observations"
    ].first().to_dict()
)

In [ ]:
# ============================================================
# FINAL MODELS 6 — FORMAT FAMILY 1 THESIS TABLE
# ============================================================

family1_term_labels = {
    "mood_NA_within_z": "Negative mood, within-person",
    "mood_NA_between_z": "Negative mood, between-person",
    "mood_PA_within_z": "Positive mood, within-person",
    "mood_PA_between_z": "Positive mood, between-person",
    "gender_female": "Female indicator",
    "family_ses_rev_z": "Socioeconomic position",
    "nwords_z": "Diary word count",
}

family1_outcome_labels = {
    "social_conn_llm": "LLM social connectedness",
    "sent_score": "Sentiment score",
    "social_density": "Social-word density",
    "neg_emotion_density": "Negative-emotion density",
}

def significance_symbol(p):
    if p < .001:
        return "***"
    if p < .01:
        return "**"
    if p < .05:
        return "*"
    if p < .10:
        return "†"
    return ""

def format_model_cell(model, term):
    beta = model.params[term]
    se = model.bse[term]
    p = model.pvalues[term]

    return (
        f"{beta:.3f} "
        f"({se:.3f})"
        f"{significance_symbol(p)}"
    )

family1_table_rows = []

for term, label in family1_term_labels.items():

    row = {"Predictor": label}

    for outcome in family1_outcomes:
        row[family1_outcome_labels[outcome]] = (
            format_model_cell(
                family1_models[outcome],
                term,
            )
        )

    family1_table_rows.append(row)

# Add model information
model_information = [
    (
        "N observations",
        lambda model: str(int(model.nobs)),
    ),
    (
        "N participants",
        lambda model: str(int(model.model.n_groups)),
    ),
    (
        "Participant intercept variance",
        lambda model: f"{float(model.cov_re.iloc[0, 0]):.3f}",
    ),
    (
        "Residual variance",
        lambda model: f"{float(model.scale):.3f}",
    ),
    (
        "ICC",
        lambda model: (
            f"{float(model.cov_re.iloc[0, 0]) / (float(model.cov_re.iloc[0, 0]) + float(model.scale)):.3f}"
        ),
    )
]

for label, value_function in model_information:

    row = {"Predictor": label}

    for outcome in family1_outcomes:
        row[family1_outcome_labels[outcome]] = (
            value_function(family1_models[outcome])
        )

    family1_table_rows.append(row)

family1_table_final = pd.DataFrame(
    family1_table_rows
)

family1_table_path = (
    TABLES / "Table_3_Family1_mood_models_common429.csv"
)

family1_table_final.to_csv(
    family1_table_path,
    index=False,
)

print("Saved:", family1_table_path)
display(family1_table_final)

## Final Model Decision 1 — Family 1 Mood Models

Family 1 was re-estimated using the common complete-case sample of 429 diary entries from 40 participants. All four language-based outcome models therefore use identical observations.

The central LLM finding is robust: diary entries recorded when participants experienced higher positive mood than usual received higher LLM social-connectedness ratings. Within-person negative mood and both between-person mood components did not clearly predict the LLM outcome.

Sentiment was primarily associated with negative mood. Higher within-person and between-person negative mood predicted lower sentiment. The small positive association between within-person positive mood and sentiment was marginally significant in the mixed model but was not supported at the .10 level under participant-clustered standard errors.

Social-word density showed no clear associations with the within-person or between-person mood components.

Within-person negative mood positively predicted negative-emotion density in the mixed model, but this result was not supported using participant-clustered robust standard errors. Because negative-emotion density is highly sparse and its mixed model produced a boundary warning, this association is treated as estimator-sensitive and is not considered a headline finding.

The principal Family 1 conclusion is therefore that diary-level positive mood is associated with LLM-rated expressed social connectedness, whereas sentiment more clearly tracks negative mood. These findings remain substantively consistent with the response memo previously reviewed by the supervisor.

## Additional Family 1 Analysis — Separate Negative- and Positive-Mood Models

These supplementary Family 1 models estimate negative mood and positive mood in separate specifications, following the supervisor's recommendation. They retain the common 429-entry sample, the same four language outcomes, the same control variables, and the same random-intercept modelling framework as the combined Family 1 models.

In [ ]:
# ============================================================
# ADDITIONAL FAMILY 1A — SET UP AND VALIDATE SEPARATE MODELS
# ============================================================

family1_separate_data = analysis.loc[
    analysis["common_language_sample"]
].copy()

family1_separate_outcomes = [
    "social_conn_llm",
    "sent_score",
    "social_density",
    "neg_emotion_density",
]

family1_separate_specs = {
    "NA_only": {
        "within": "mood_NA_within_z",
        "between": "mood_NA_between_z",
    },
    "PA_only": {
        "within": "mood_PA_within_z",
        "between": "mood_PA_between_z",
    },
}

family1_separate_controls = [
    "gender_female",
    "family_ses_rev_z",
    "nwords_z",
]

required_columns = [
    "participantid",
    *family1_separate_outcomes,
    *family1_separate_controls,
    "mood_NA_within_z",
    "mood_NA_between_z",
    "mood_PA_within_z",
    "mood_PA_between_z",
]

missing_columns = [
    column
    for column in required_columns
    if column not in family1_separate_data.columns
]

missing_values = int(
    family1_separate_data[
        required_columns
    ].isna().sum().sum()
)

print("Rows:", len(family1_separate_data))
print(
    "Participants:",
    family1_separate_data["participantid"].nunique(),
)
print("Missing columns:", missing_columns)
print("Missing values:", missing_values)

assert len(family1_separate_data) == 429
assert (
    family1_separate_data["participantid"].nunique()
    == 40
)
assert missing_columns == []
assert missing_values == 0

print(
    "\nValidation passed: separate Family 1 models "
    "will use 429 entries from 40 participants."
)

In [ ]:
# ============================================================
# ADDITIONAL FAMILY 1B — DEFINE MIXED-MODEL FITTING FUNCTION
# ============================================================

def fit_separate_mixed_model(
    formula,
    data,
    group_column="participantid",
):
    optimizers = [
        "lbfgs",
        "bfgs",
        "cg",
        "powell",
        "nm",
    ]

    attempts = []

    for optimizer in optimizers:
        try:
            with warnings.catch_warnings(record=True) as caught:
                warnings.simplefilter("always")

                candidate = smf.mixedlm(
                    formula=formula,
                    data=data,
                    groups=data[group_column],
                ).fit(
                    reml=False,
                    method=optimizer,
                    disp=False,
                )

            warning_messages = [
                str(item.message)
                for item in caught
            ]

            attempts.append({
                "optimizer": optimizer,
                "converged": bool(candidate.converged),
                "warnings": warning_messages,
                "error": None,
            })

            if candidate.converged:
                return {
                    "model": candidate,
                    "optimizer": optimizer,
                    "warnings": warning_messages,
                    "attempts": attempts,
                }

        except Exception as error:
            attempts.append({
                "optimizer": optimizer,
                "converged": False,
                "warnings": [],
                "error": (
                    f"{type(error).__name__}: {error}"
                ),
            })

    raise RuntimeError(
        "The mixed model did not converge with any "
        f"optimizer. Attempts: {attempts}"
    )


print(
    "Separate mixed-model fitting function "
    "defined successfully."
)

In [ ]:
# ============================================================
# ADDITIONAL FAMILY 1C — TEST ONE SEPARATE MIXED MODEL
# ============================================================

test_formula = (
    "social_conn_llm ~ "
    "mood_NA_within_z + "
    "mood_NA_between_z + "
    "gender_female + "
    "family_ses_rev_z + "
    "nwords_z"
)

test_result = fit_separate_mixed_model(
    formula=test_formula,
    data=family1_separate_data,
)

test_model = test_result["model"]

print("Formula:", test_formula)
print("Optimizer used:", test_result["optimizer"])
print("Converged:", test_model.converged)
print("Observations:", int(test_model.nobs))
print(
    "Participants:",
    int(test_model.model.n_groups),
)
print(
    "Participant intercept variance:",
    float(test_model.cov_re.iloc[0, 0]),
)
print(
    "Residual variance:",
    float(test_model.scale),
)
print(
    "Warnings:",
    test_result["warnings"],
)

assert test_model.converged
assert int(test_model.nobs) == 429
assert int(test_model.model.n_groups) == 40

print("\nTest model passed all checks.")

In [ ]:
# Inspect all optimizer attempts for the test model

for attempt in test_result["attempts"]:
    print("Optimizer:", attempt["optimizer"])
    print("Converged:", attempt["converged"])
    print("Warnings:", attempt["warnings"])
    print("Error:", attempt["error"])
    print("-" * 50)

In [ ]:
# ============================================================
# ADDITIONAL FAMILY 1D — CHECK TEST MODEL WITH BFGS
# ============================================================

with warnings.catch_warnings(record=True) as caught_bfgs:
    warnings.simplefilter("always")

    test_model_bfgs = smf.mixedlm(
        formula=test_formula,
        data=family1_separate_data,
        groups=family1_separate_data["participantid"],
    ).fit(
        reml=False,
        method="bfgs",
        disp=False,
    )

bfgs_warnings = [
    str(item.message)
    for item in caught_bfgs
]

print("Optimizer: bfgs")
print("Converged:", test_model_bfgs.converged)
print("Observations:", int(test_model_bfgs.nobs))
print(
    "Participants:",
    int(test_model_bfgs.model.n_groups),
)
print(
    "Participant intercept variance:",
    float(test_model_bfgs.cov_re.iloc[0, 0]),
)
print(
    "Residual variance:",
    float(test_model_bfgs.scale),
)
print("Log-likelihood:", float(test_model_bfgs.llf))
print("Warnings:", bfgs_warnings)

assert test_model_bfgs.converged
assert int(test_model_bfgs.nobs) == 429
assert int(test_model_bfgs.model.n_groups) == 40

print("\nBFGS test completed.")

In [ ]:
# Store the verified test model safely

family1_separate_models = {
    "NA_only_social_conn_llm": test_model_bfgs
}

family1_separate_model_info = {
    "NA_only_social_conn_llm": {
        "optimizer": "bfgs",
        "warnings": bfgs_warnings,
    }
}

print(
    "Stored models:",
    list(family1_separate_models.keys()),
)

print(
    "Stored optimizer:",
    family1_separate_model_info[
        "NA_only_social_conn_llm"
    ]["optimizer"],
)

print(
    "Stored warnings:",
    family1_separate_model_info[
        "NA_only_social_conn_llm"
    ]["warnings"],
)

In [ ]:
comparison_rows = []

for term in test_model_bfgs.fe_params.index:
    comparison_rows.append({
        "term": term,
        "lbfgs_beta": test_model.fe_params[term],
        "bfgs_beta": test_model_bfgs.fe_params[term],
        "absolute_beta_difference": abs(
            test_model.fe_params[term]
            - test_model_bfgs.fe_params[term]
        ),
        "lbfgs_SE": test_model.bse[term],
        "bfgs_SE": test_model_bfgs.bse[term],
        "absolute_SE_difference": abs(
            test_model.bse[term]
            - test_model_bfgs.bse[term]
        ),
        "lbfgs_p": test_model.pvalues[term],
        "bfgs_p": test_model_bfgs.pvalues[term],
    })

test_optimizer_comparison = pd.DataFrame(
    comparison_rows
)

display(
    test_optimizer_comparison.round(8)
)

print(
    "L-BFGS log-likelihood:",
    float(test_model.llf),
)

print(
    "BFGS log-likelihood:",
    float(test_model_bfgs.llf),
)

print(
    "Absolute log-likelihood difference:",
    abs(
        float(test_model.llf)
        - float(test_model_bfgs.llf)
    ),
)

print(
    "Maximum absolute coefficient difference:",
    test_optimizer_comparison[
        "absolute_beta_difference"
    ].max(),
)

print(
    "Maximum absolute SE difference:",
    test_optimizer_comparison[
        "absolute_SE_difference"
    ].max(),
)

In [ ]:
print("L-BFGS log-likelihood:", float(test_model.llf))
print("BFGS log-likelihood:", float(test_model_bfgs.llf))

print(
    "Absolute log-likelihood difference:",
    abs(
        float(test_model.llf)
        - float(test_model_bfgs.llf)
    ),
)

In [ ]:
def fit_final_separate_model(formula, data):

    optimizers = [
        "bfgs",
        "lbfgs",
        "cg",
        "powell",
        "nm",
    ]

    attempts = []

    for optimizer in optimizers:

        try:
            with warnings.catch_warnings(record=True) as caught:
                warnings.simplefilter("always")

                model = smf.mixedlm(
                    formula=formula,
                    data=data,
                    groups=data["participantid"],
                ).fit(
                    reml=False,
                    method=optimizer,
                    disp=False,
                )

            warning_messages = [
                str(item.message)
                for item in caught
            ]

            problematic_warning = any(
                phrase in warning.lower()
                for warning in warning_messages
                for phrase in [
                    "singular",
                    "boundary",
                    "failed to converge",
                ]
            )

            attempts.append({
                "optimizer": optimizer,
                "converged": bool(model.converged),
                "warnings": warning_messages,
                "problematic_warning": problematic_warning,
                "error": None,
            })

            if model.converged and not problematic_warning:
                return {
                    "model": model,
                    "optimizer": optimizer,
                    "warnings": warning_messages,
                    "attempts": attempts,
                    "clean_fit": True,
                }

        except Exception as error:

            attempts.append({
                "optimizer": optimizer,
                "converged": False,
                "warnings": [],
                "problematic_warning": True,
                "error": (
                    f"{type(error).__name__}: {error}"
                ),
            })

    raise RuntimeError(
        "No converged warning-free solution was found. "
        f"Attempts: {attempts}"
    )


print("Final separate-model fitting function defined.")

In [ ]:
# ============================================================
# ADDITIONAL FAMILY 1E — FIT NA-ONLY SENTIMENT MODEL
# ============================================================

na_sentiment_formula = (
    "sent_score ~ "
    "mood_NA_within_z + "
    "mood_NA_between_z + "
    "gender_female + "
    "family_ses_rev_z + "
    "nwords_z"
)

na_sentiment_result = fit_final_separate_model(
    formula=na_sentiment_formula,
    data=family1_separate_data,
)

na_sentiment_model = na_sentiment_result["model"]

print("Model ID: NA_only_sent_score")
print(
    "Optimizer:",
    na_sentiment_result["optimizer"],
)
print(
    "Converged:",
    na_sentiment_model.converged,
)
print(
    "Observations:",
    int(na_sentiment_model.nobs),
)
print(
    "Participants:",
    int(na_sentiment_model.model.n_groups),
)
print(
    "Participant intercept variance:",
    float(na_sentiment_model.cov_re.iloc[0, 0]),
)
print(
    "Residual variance:",
    float(na_sentiment_model.scale),
)
print(
    "Warnings:",
    na_sentiment_result["warnings"],
)

print("\nFocal estimates:")

for term in [
    "mood_NA_within_z",
    "mood_NA_between_z",
]:
    print(
        term,
        "beta =",
        round(
            float(na_sentiment_model.fe_params[term]),
            6,
        ),
        "SE =",
        round(
            float(na_sentiment_model.bse[term]),
            6,
        ),
        "p =",
        round(
            float(na_sentiment_model.pvalues[term]),
            6,
        ),
    )

assert na_sentiment_model.converged
assert int(na_sentiment_model.nobs) == 429
assert int(na_sentiment_model.model.n_groups) == 40

print("\nNA-only sentiment model passed.")

In [ ]:
with warnings.catch_warnings(record=True) as caught_powell:
    warnings.simplefilter("always")

    na_sentiment_powell = smf.mixedlm(
        formula=na_sentiment_formula,
        data=family1_separate_data,
        groups=family1_separate_data["participantid"],
    ).fit(
        reml=False,
        method="powell",
        disp=False,
    )

powell_warnings = [
    str(item.message)
    for item in caught_powell
]

participant_variance = float(
    na_sentiment_powell.cov_re.iloc[0, 0]
)

residual_variance = float(
    na_sentiment_powell.scale
)

icc = (
    participant_variance
    / (participant_variance + residual_variance)
)

print("Optimizer: powell")
print("Converged:", na_sentiment_powell.converged)
print("Observations:", int(na_sentiment_powell.nobs))
print(
    "Participants:",
    int(na_sentiment_powell.model.n_groups),
)
print(
    "Participant intercept variance:",
    participant_variance,
)
print("Residual variance:", residual_variance)
print("ICC:", icc)
print("Log-likelihood:", float(na_sentiment_powell.llf))
print("Warnings:", powell_warnings)

for term in [
    "mood_NA_within_z",
    "mood_NA_between_z",
]:
    print(
        term,
        "beta =",
        float(na_sentiment_powell.fe_params[term]),
        "SE =",
        float(na_sentiment_powell.bse[term]),
        "p =",
        float(na_sentiment_powell.pvalues[term]),
    )

In [ ]:
with warnings.catch_warnings(record=True) as caught_nm:
    warnings.simplefilter("always")

    na_sentiment_nm = smf.mixedlm(
        formula=na_sentiment_formula,
        data=family1_separate_data,
        groups=family1_separate_data["participantid"],
    ).fit(
        reml=False,
        method="nm",
        disp=False,
    )

nm_warnings = [
    str(item.message)
    for item in caught_nm
]

nm_participant_variance = float(
    na_sentiment_nm.cov_re.iloc[0, 0]
)

nm_residual_variance = float(
    na_sentiment_nm.scale
)

nm_icc = (
    nm_participant_variance
    / (
        nm_participant_variance
        + nm_residual_variance
    )
)

print("Optimizer: nm")
print("Converged:", na_sentiment_nm.converged)

print(
    "Participant intercept variance:",
    nm_participant_variance,
)

print(
    "Residual variance:",
    nm_residual_variance,
)

print("ICC:", nm_icc)

print(
    "Log-likelihood:",
    float(na_sentiment_nm.llf),
)

print("Warnings:", nm_warnings)

for term in [
    "mood_NA_within_z",
    "mood_NA_between_z",
]:
    print(
        term,
        "beta =",
        float(na_sentiment_nm.fe_params[term]),
        "SE =",
        float(na_sentiment_nm.bse[term]),
        "p =",
        float(na_sentiment_nm.pvalues[term]),
    )

In [ ]:
na_sentiment_cluster = smf.ols(
    formula=na_sentiment_formula,
    data=family1_separate_data,
).fit(
    cov_type="cluster",
    cov_kwds={
        "groups": family1_separate_data["participantid"]
    },
)

print("Clustered OLS observations:", int(na_sentiment_cluster.nobs))

for term in [
    "mood_NA_within_z",
    "mood_NA_between_z",
]:
    print(
        term,
        "beta =",
        float(na_sentiment_cluster.params[term]),
        "SE =",
        float(na_sentiment_cluster.bse[term]),
        "p =",
        float(na_sentiment_cluster.pvalues[term]),
    )

In [ ]:
# Store the verified NA-only sentiment results

family1_separate_models[
    "NA_only_sent_score"
] = na_sentiment_powell

family1_separate_model_info[
    "NA_only_sent_score"
] = {
    "optimizer": "powell",
    "warnings": powell_warnings,
    "boundary_warning": True,
    "participant_intercept_variance": float(
        na_sentiment_powell.cov_re.iloc[0, 0]
    ),
    "residual_variance": float(
        na_sentiment_powell.scale
    ),
    "ICC": float(icc),
}

family1_separate_cluster_models = {
    "NA_only_sent_score": na_sentiment_cluster
}

print(
    "Stored mixed models:",
    list(family1_separate_models.keys()),
)

print(
    "Stored clustered models:",
    list(family1_separate_cluster_models.keys()),
)

print(
    "NA-only sentiment optimizer:",
    family1_separate_model_info[
        "NA_only_sent_score"
    ]["optimizer"],
)

print(
    "NA-only sentiment boundary warning:",
    family1_separate_model_info[
        "NA_only_sent_score"
    ]["boundary_warning"],
)

In [ ]:
pa_llm_formula = (
    "social_conn_llm ~ "
    "mood_PA_within_z + "
    "mood_PA_between_z + "
    "gender_female + "
    "family_ses_rev_z + "
    "nwords_z"
)

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")

    pa_llm_bfgs = smf.mixedlm(
        formula=pa_llm_formula,
        data=family1_separate_data,
        groups=family1_separate_data["participantid"],
    ).fit(
        reml=False,
        method="bfgs",
        disp=False,
    )

pa_llm_warnings = [
    str(item.message)
    for item in caught
]

participant_variance = float(
    pa_llm_bfgs.cov_re.iloc[0, 0]
)

residual_variance = float(
    pa_llm_bfgs.scale
)

pa_llm_icc = (
    participant_variance
    / (participant_variance + residual_variance)
)

print("Model ID: PA_only_social_conn_llm")
print("Optimizer: bfgs")
print("Converged:", pa_llm_bfgs.converged)
print("Observations:", int(pa_llm_bfgs.nobs))
print("Participants:", int(pa_llm_bfgs.model.n_groups))
print("Participant intercept variance:", participant_variance)
print("Residual variance:", residual_variance)
print("ICC:", pa_llm_icc)
print("Log-likelihood:", float(pa_llm_bfgs.llf))
print("Warnings:", pa_llm_warnings)

for term in [
    "mood_PA_within_z",
    "mood_PA_between_z",
]:
    print(
        term,
        "beta =",
        float(pa_llm_bfgs.fe_params[term]),
        "SE =",
        float(pa_llm_bfgs.bse[term]),
        "p =",
        float(pa_llm_bfgs.pvalues[term]),
    )

In [ ]:
pa_llm_cluster = smf.ols(
    formula=pa_llm_formula,
    data=family1_separate_data,
).fit(
    cov_type="cluster",
    cov_kwds={
        "groups": family1_separate_data["participantid"]
    },
)

print(
    "Clustered OLS observations:",
    int(pa_llm_cluster.nobs),
)

for term in [
    "mood_PA_within_z",
    "mood_PA_between_z",
]:
    print(
        term,
        "beta =",
        float(pa_llm_cluster.params[term]),
        "SE =",
        float(pa_llm_cluster.bse[term]),
        "p =",
        float(pa_llm_cluster.pvalues[term]),
    )

In [ ]:
# Store the verified PA-only LLM results

family1_separate_models[
    "PA_only_social_conn_llm"
] = pa_llm_bfgs

family1_separate_model_info[
    "PA_only_social_conn_llm"
] = {
    "optimizer": "bfgs",
    "warnings": pa_llm_warnings,
    "boundary_warning": False,
    "participant_intercept_variance": float(
        pa_llm_bfgs.cov_re.iloc[0, 0]
    ),
    "residual_variance": float(
        pa_llm_bfgs.scale
    ),
    "ICC": float(pa_llm_icc),
}

family1_separate_cluster_models[
    "PA_only_social_conn_llm"
] = pa_llm_cluster

print(
    "Stored mixed models:",
    list(family1_separate_models.keys()),
)

print(
    "Stored clustered models:",
    list(family1_separate_cluster_models.keys()),
)

print(
    "PA-only LLM optimizer:",
    family1_separate_model_info[
        "PA_only_social_conn_llm"
    ]["optimizer"],
)

print(
    "PA-only LLM boundary warning:",
    family1_separate_model_info[
        "PA_only_social_conn_llm"
    ]["boundary_warning"],
)

In [ ]:
# ============================================================
# ADDITIONAL FAMILY 1F — DIAGNOSE PA-ONLY SENTIMENT MODEL
# ============================================================

pa_sentiment_formula = (
    "sent_score ~ "
    "mood_PA_within_z + "
    "mood_PA_between_z + "
    "gender_female + "
    "family_ses_rev_z + "
    "nwords_z"
)

pa_sentiment_candidates = {}
pa_sentiment_diagnostics = []

for optimizer in [
    "bfgs",
    "lbfgs",
    "cg",
    "powell",
    "nm",
]:
    try:
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")

            model = smf.mixedlm(
                formula=pa_sentiment_formula,
                data=family1_separate_data,
                groups=family1_separate_data["participantid"],
            ).fit(
                reml=False,
                method=optimizer,
                disp=False,
            )

        warning_messages = [
            str(item.message)
            for item in caught
        ]

        pa_sentiment_candidates[optimizer] = model

        pa_sentiment_diagnostics.append({
            "optimizer": optimizer,
            "converged": bool(model.converged),
            "intercept_variance": float(
                model.cov_re.iloc[0, 0]
            ),
            "residual_variance": float(model.scale),
            "log_likelihood": float(model.llf),
            "warnings": " | ".join(warning_messages),
            "error": "",
        })

    except Exception as error:
        pa_sentiment_diagnostics.append({
            "optimizer": optimizer,
            "converged": False,
            "intercept_variance": np.nan,
            "residual_variance": np.nan,
            "log_likelihood": np.nan,
            "warnings": "",
            "error": (
                f"{type(error).__name__}: {error}"
            ),
        })

pa_sentiment_diagnostics = pd.DataFrame(
    pa_sentiment_diagnostics
)

display(pa_sentiment_diagnostics)

In [ ]:
pa_sentiment_cg = pa_sentiment_candidates["cg"]

pa_sentiment_participant_variance = float(
    pa_sentiment_cg.cov_re.iloc[0, 0]
)

pa_sentiment_residual_variance = float(
    pa_sentiment_cg.scale
)

pa_sentiment_icc = (
    pa_sentiment_participant_variance
    / (
        pa_sentiment_participant_variance
        + pa_sentiment_residual_variance
    )
)

print("Model ID: PA_only_sent_score")
print("Optimizer: cg")
print("Converged:", pa_sentiment_cg.converged)
print("Observations:", int(pa_sentiment_cg.nobs))
print(
    "Participants:",
    int(pa_sentiment_cg.model.n_groups),
)
print(
    "Participant intercept variance:",
    pa_sentiment_participant_variance,
)
print(
    "Residual variance:",
    pa_sentiment_residual_variance,
)
print("ICC:", pa_sentiment_icc)
print(
    "Log-likelihood:",
    float(pa_sentiment_cg.llf),
)

for term in [
    "mood_PA_within_z",
    "mood_PA_between_z",
]:
    print(
        term,
        "beta =",
        float(pa_sentiment_cg.fe_params[term]),
        "SE =",
        float(pa_sentiment_cg.bse[term]),
        "p =",
        float(pa_sentiment_cg.pvalues[term]),
    )


In [ ]:
pa_sentiment_cluster = smf.ols(
    formula=pa_sentiment_formula,
    data=family1_separate_data,
).fit(
    cov_type="cluster",
    cov_kwds={
        "groups": family1_separate_data["participantid"]
    },
)

print(
    "Clustered OLS observations:",
    int(pa_sentiment_cluster.nobs),
)

for term in [
    "mood_PA_within_z",
    "mood_PA_between_z",
]:
    print(
        term,
        "beta =",
        float(pa_sentiment_cluster.params[term]),
        "SE =",
        float(pa_sentiment_cluster.bse[term]),
        "p =",
        float(pa_sentiment_cluster.pvalues[term]),
    )

In [ ]:
family1_separate_models[
    "PA_only_sent_score"
] = pa_sentiment_cg

family1_separate_model_info[
    "PA_only_sent_score"
] = {
    "optimizer": "cg",
    "warnings": [],
    "boundary_warning": False,
    "participant_intercept_variance": float(
        pa_sentiment_cg.cov_re.iloc[0, 0]
    ),
    "residual_variance": float(
        pa_sentiment_cg.scale
    ),
    "ICC": float(pa_sentiment_icc),
}

family1_separate_cluster_models[
    "PA_only_sent_score"
] = pa_sentiment_cluster

print(
    "Stored mixed models:",
    list(family1_separate_models.keys()),
)

print(
    "Stored clustered models:",
    list(family1_separate_cluster_models.keys()),
)

print(
    "PA-only sentiment optimizer:",
    family1_separate_model_info[
        "PA_only_sent_score"
    ]["optimizer"],
)

print(
    "PA-only sentiment boundary warning:",
    family1_separate_model_info[
        "PA_only_sent_score"
    ]["boundary_warning"],
)

In [ ]:
na_llm_cluster = smf.ols(
    formula=test_formula,
    data=family1_separate_data,
).fit(
    cov_type="cluster",
    cov_kwds={
        "groups": family1_separate_data["participantid"]
    },
)

print(
    "Clustered OLS observations:",
    int(na_llm_cluster.nobs),
)

for term in [
    "mood_NA_within_z",
    "mood_NA_between_z",
]:
    print(
        term,
        "beta =",
        float(na_llm_cluster.params[term]),
        "SE =",
        float(na_llm_cluster.bse[term]),
        "p =",
        float(na_llm_cluster.pvalues[term]),
    )

In [ ]:
family1_separate_cluster_models[
    "NA_only_social_conn_llm"
] = na_llm_cluster

print(
    "Stored clustered models:",
    list(family1_separate_cluster_models.keys()),
)

print(
    "NA-only LLM clustered observations:",
    int(
        family1_separate_cluster_models[
            "NA_only_social_conn_llm"
        ].nobs
    ),
)

In [ ]:
# NA-only social-word-density formula

na_social_density_formula = (
    "social_density ~ "
    "mood_NA_within_z + "
    "mood_NA_between_z + "
    "gender_female + "
    "family_ses_rev_z + "
    "nwords_z"
)

na_social_density_diagnostics = []

for optimizer in [
    "bfgs",
    "lbfgs",
    "cg",
    "powell",
    "nm",
]:
    try:
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")

            model = smf.mixedlm(
                formula=na_social_density_formula,
                data=family1_separate_data,
                groups=family1_separate_data[
                    "participantid"
                ],
            ).fit(
                reml=False,
                method=optimizer,
                disp=False,
            )

        warning_text = " | ".join(
            str(item.message)
            for item in caught
        )

        na_social_density_diagnostics.append({
            "optimizer": optimizer,
            "converged": bool(model.converged),
            "intercept_variance": float(
                model.cov_re.iloc[0, 0]
            ),
            "residual_variance": float(model.scale),
            "log_likelihood": float(model.llf),
            "warnings": warning_text,
            "error": "",
        })

    except Exception as error:
        na_social_density_diagnostics.append({
            "optimizer": optimizer,
            "converged": False,
            "intercept_variance": np.nan,
            "residual_variance": np.nan,
            "log_likelihood": np.nan,
            "warnings": "",
            "error": (
                f"{type(error).__name__}: {error}"
            ),
        })

na_social_density_diagnostics = pd.DataFrame(
    na_social_density_diagnostics
)

display(na_social_density_diagnostics)

In [ ]:
na_social_density_bfgs = smf.mixedlm(
    formula=na_social_density_formula,
    data=family1_separate_data,
    groups=family1_separate_data["participantid"],
)

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")

    na_social_density_bfgs = (
        na_social_density_bfgs.fit(
            reml=False,
            method="bfgs",
            disp=False,
        )
    )

na_social_density_warnings = [
    str(item.message)
    for item in caught
]

na_social_density_intercept_var = float(
    na_social_density_bfgs.cov_re.iloc[0, 0]
)

na_social_density_residual_var = float(
    na_social_density_bfgs.scale
)

na_social_density_icc = (
    na_social_density_intercept_var
    / (
        na_social_density_intercept_var
        + na_social_density_residual_var
    )
)

print("Model ID: NA_only_social_density")
print("Optimizer: bfgs")
print("Converged:", na_social_density_bfgs.converged)
print("Observations:", int(na_social_density_bfgs.nobs))
print(
    "Participants:",
    int(na_social_density_bfgs.model.n_groups),
)
print(
    "Participant intercept variance:",
    na_social_density_intercept_var,
)
print(
    "Residual variance:",
    na_social_density_residual_var,
)
print("ICC:", na_social_density_icc)
print(
    "Log-likelihood:",
    float(na_social_density_bfgs.llf),
)
print(
    "Warnings:",
    na_social_density_warnings,
)

for term in [
    "mood_NA_within_z",
    "mood_NA_between_z",
]:
    print(
        term,
        "beta =",
        float(na_social_density_bfgs.fe_params[term]),
        "SE =",
        float(na_social_density_bfgs.bse[term]),
        "p =",
        float(na_social_density_bfgs.pvalues[term]),
    )

In [ ]:
na_social_density_cluster = smf.ols(
    formula=na_social_density_formula,
    data=family1_separate_data,
).fit(
    cov_type="cluster",
    cov_kwds={
        "groups": family1_separate_data["participantid"]
    },
)

print(
    "Clustered OLS observations:",
    int(na_social_density_cluster.nobs),
)

for term in [
    "mood_NA_within_z",
    "mood_NA_between_z",
]:
    print(
        term,
        "beta =",
        float(na_social_density_cluster.params[term]),
        "SE =",
        float(na_social_density_cluster.bse[term]),
        "p =",
        float(na_social_density_cluster.pvalues[term]),
    )

In [ ]:
family1_separate_models[
    "NA_only_social_density"
] = na_social_density_bfgs

family1_separate_model_info[
    "NA_only_social_density"
] = {
    "optimizer": "bfgs",
    "warnings": na_social_density_warnings,
    "boundary_warning": True,
    "participant_intercept_variance": float(
        na_social_density_bfgs.cov_re.iloc[0, 0]
    ),
    "residual_variance": float(
        na_social_density_bfgs.scale
    ),
    "ICC": float(na_social_density_icc),
}

family1_separate_cluster_models[
    "NA_only_social_density"
] = na_social_density_cluster

print(
    "Stored mixed models:",
    list(family1_separate_models.keys()),
)

print(
    "Stored clustered models:",
    list(family1_separate_cluster_models.keys()),
)

print(
    "NA-only social-density optimizer:",
    family1_separate_model_info[
        "NA_only_social_density"
    ]["optimizer"],
)

print(
    "NA-only social-density boundary warning:",
    family1_separate_model_info[
        "NA_only_social_density"
    ]["boundary_warning"],
)

In [ ]:
pa_social_density_formula = (
    "social_density ~ "
    "mood_PA_within_z + "
    "mood_PA_between_z + "
    "gender_female + "
    "family_ses_rev_z + "
    "nwords_z"
)

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")

    pa_social_density_bfgs = smf.mixedlm(
        formula=pa_social_density_formula,
        data=family1_separate_data,
        groups=family1_separate_data["participantid"],
    ).fit(
        reml=False,
        method="bfgs",
        disp=False,
    )

pa_social_density_warnings = [
    str(item.message)
    for item in caught
]

pa_social_density_intercept_var = float(
    pa_social_density_bfgs.cov_re.iloc[0, 0]
)

pa_social_density_residual_var = float(
    pa_social_density_bfgs.scale
)

pa_social_density_icc = (
    pa_social_density_intercept_var
    / (
        pa_social_density_intercept_var
        + pa_social_density_residual_var
    )
)

print("Model ID: PA_only_social_density")
print("Optimizer: bfgs")
print("Converged:", pa_social_density_bfgs.converged)
print("Observations:", int(pa_social_density_bfgs.nobs))
print("Participants:", int(pa_social_density_bfgs.model.n_groups))
print("Participant intercept variance:", pa_social_density_intercept_var)
print("Residual variance:", pa_social_density_residual_var)
print("ICC:", pa_social_density_icc)
print("Log-likelihood:", float(pa_social_density_bfgs.llf))
print("Warnings:", pa_social_density_warnings)

for term in [
    "mood_PA_within_z",
    "mood_PA_between_z",
]:
    print(
        term,
        "beta =",
        float(pa_social_density_bfgs.fe_params[term]),
        "SE =",
        float(pa_social_density_bfgs.bse[term]),
        "p =",
        float(pa_social_density_bfgs.pvalues[term]),
    )

In [ ]:
pa_social_density_cluster = smf.ols(
    formula=pa_social_density_formula,
    data=family1_separate_data,
).fit(
    cov_type="cluster",
    cov_kwds={
        "groups": family1_separate_data["participantid"]
    },
)

print(
    "Clustered OLS observations:",
    int(pa_social_density_cluster.nobs),
)

for term in [
    "mood_PA_within_z",
    "mood_PA_between_z",
]:
    print(
        term,
        "beta =",
        float(pa_social_density_cluster.params[term]),
        "SE =",
        float(pa_social_density_cluster.bse[term]),
        "p =",
        float(pa_social_density_cluster.pvalues[term]),
    )

In [ ]:
family1_separate_models[
    "PA_only_social_density"
] = pa_social_density_bfgs

family1_separate_model_info[
    "PA_only_social_density"
] = {
    "optimizer": "bfgs",
    "warnings": pa_social_density_warnings,
    "boundary_warning": True,
    "participant_intercept_variance": float(
        pa_social_density_bfgs.cov_re.iloc[0, 0]
    ),
    "residual_variance": float(
        pa_social_density_bfgs.scale
    ),
    "ICC": float(pa_social_density_icc),
}

family1_separate_cluster_models[
    "PA_only_social_density"
] = pa_social_density_cluster

print(
    "Number of stored mixed models:",
    len(family1_separate_models),
)

print(
    "Number of stored clustered models:",
    len(family1_separate_cluster_models),
)

print(
    "PA-only social-density optimizer:",
    family1_separate_model_info[
        "PA_only_social_density"
    ]["optimizer"],
)

print(
    "PA-only social-density boundary warning:",
    family1_separate_model_info[
        "PA_only_social_density"
    ]["boundary_warning"],
)

In [ ]:
na_neg_emotion_formula = (
    "neg_emotion_density ~ "
    "mood_NA_within_z + "
    "mood_NA_between_z + "
    "gender_female + "
    "family_ses_rev_z + "
    "nwords_z"
)

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")

    na_neg_emotion_bfgs = smf.mixedlm(
        formula=na_neg_emotion_formula,
        data=family1_separate_data,
        groups=family1_separate_data["participantid"],
    ).fit(
        reml=False,
        method="bfgs",
        disp=False,
    )

na_neg_emotion_warnings = [
    str(item.message)
    for item in caught
]

na_neg_emotion_intercept_var = float(
    na_neg_emotion_bfgs.cov_re.iloc[0, 0]
)

na_neg_emotion_residual_var = float(
    na_neg_emotion_bfgs.scale
)

na_neg_emotion_icc = (
    na_neg_emotion_intercept_var
    / (
        na_neg_emotion_intercept_var
        + na_neg_emotion_residual_var
    )
)

print("Model ID: NA_only_neg_emotion_density")
print("Optimizer: bfgs")
print("Converged:", na_neg_emotion_bfgs.converged)
print("Observations:", int(na_neg_emotion_bfgs.nobs))
print("Participants:", int(na_neg_emotion_bfgs.model.n_groups))
print("Participant intercept variance:", na_neg_emotion_intercept_var)
print("Residual variance:", na_neg_emotion_residual_var)
print("ICC:", na_neg_emotion_icc)
print("Log-likelihood:", float(na_neg_emotion_bfgs.llf))
print("Warnings:", na_neg_emotion_warnings)

for term in [
    "mood_NA_within_z",
    "mood_NA_between_z",
]:
    print(
        term,
        "beta =",
        float(na_neg_emotion_bfgs.fe_params[term]),
        "SE =",
        float(na_neg_emotion_bfgs.bse[term]),
        "p =",
        float(na_neg_emotion_bfgs.pvalues[term]),
    )

In [ ]:
na_neg_emotion_candidates = {}
na_neg_emotion_diagnostics = []

for optimizer in ["lbfgs", "cg", "powell", "nm"]:
    try:
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")

            model = smf.mixedlm(
                formula=na_neg_emotion_formula,
                data=family1_separate_data,
                groups=family1_separate_data["participantid"],
            ).fit(
                reml=False,
                method=optimizer,
                disp=False,
            )

        warning_messages = [
            str(item.message)
            for item in caught
        ]

        na_neg_emotion_candidates[optimizer] = model

        na_neg_emotion_diagnostics.append({
            "optimizer": optimizer,
            "converged": bool(model.converged),
            "intercept_variance": float(
                model.cov_re.iloc[0, 0]
            ),
            "residual_variance": float(model.scale),
            "log_likelihood": float(model.llf),
            "warnings": " | ".join(warning_messages),
            "error": "",
        })

    except Exception as error:
        na_neg_emotion_diagnostics.append({
            "optimizer": optimizer,
            "converged": False,
            "intercept_variance": np.nan,
            "residual_variance": np.nan,
            "log_likelihood": np.nan,
            "warnings": "",
            "error": (
                f"{type(error).__name__}: {error}"
            ),
        })

na_neg_emotion_diagnostics = pd.DataFrame(
    na_neg_emotion_diagnostics
)

display(na_neg_emotion_diagnostics)

In [ ]:
na_neg_emotion_powell = (
    na_neg_emotion_candidates["powell"]
)

na_neg_emotion_powell_intercept_var = float(
    na_neg_emotion_powell.cov_re.iloc[0, 0]
)

na_neg_emotion_powell_residual_var = float(
    na_neg_emotion_powell.scale
)

na_neg_emotion_powell_icc = (
    na_neg_emotion_powell_intercept_var
    / (
        na_neg_emotion_powell_intercept_var
        + na_neg_emotion_powell_residual_var
    )
)

print("Model ID: NA_only_neg_emotion_density")
print("Optimizer: powell")
print("Converged:", na_neg_emotion_powell.converged)

print(
    "Observations:",
    int(na_neg_emotion_powell.nobs),
)

print(
    "Participants:",
    int(na_neg_emotion_powell.model.n_groups),
)

print(
    "Participant intercept variance:",
    na_neg_emotion_powell_intercept_var,
)

print(
    "Residual variance:",
    na_neg_emotion_powell_residual_var,
)

print("ICC:", na_neg_emotion_powell_icc)

print(
    "Log-likelihood:",
    float(na_neg_emotion_powell.llf),
)

for term in [
    "mood_NA_within_z",
    "mood_NA_between_z",
]:
    print(
        term,
        "beta =",
        float(na_neg_emotion_powell.fe_params[term]),
        "SE =",
        float(na_neg_emotion_powell.bse[term]),
        "p =",
        float(na_neg_emotion_powell.pvalues[term]),
    )

In [ ]:
na_neg_emotion_cluster = smf.ols(
    formula=na_neg_emotion_formula,
    data=family1_separate_data,
).fit(
    cov_type="cluster",
    cov_kwds={
        "groups": family1_separate_data["participantid"]
    },
)

print(
    "Clustered OLS observations:",
    int(na_neg_emotion_cluster.nobs),
)

for term in [
    "mood_NA_within_z",
    "mood_NA_between_z",
]:
    print(
        term,
        "beta =",
        float(na_neg_emotion_cluster.params[term]),
        "SE =",
        float(na_neg_emotion_cluster.bse[term]),
        "p =",
        float(na_neg_emotion_cluster.pvalues[term]),
    )

In [ ]:
family1_separate_models[
    "NA_only_neg_emotion_density"
] = na_neg_emotion_powell

family1_separate_model_info[
    "NA_only_neg_emotion_density"
] = {
    "optimizer": "powell",
    "warnings": [
        "The MLE may be on the boundary "
        "of the parameter space."
    ],
    "boundary_warning": True,
    "estimator_sensitive": True,
    "participant_intercept_variance": float(
        na_neg_emotion_powell.cov_re.iloc[0, 0]
    ),
    "residual_variance": float(
        na_neg_emotion_powell.scale
    ),
    "ICC": float(
        na_neg_emotion_powell_icc
    ),
}

family1_separate_cluster_models[
    "NA_only_neg_emotion_density"
] = na_neg_emotion_cluster

print(
    "Number of stored mixed models:",
    len(family1_separate_models),
)

print(
    "Number of stored clustered models:",
    len(family1_separate_cluster_models),
)

print(
    "Optimizer:",
    family1_separate_model_info[
        "NA_only_neg_emotion_density"
    ]["optimizer"],
)

print(
    "Boundary warning:",
    family1_separate_model_info[
        "NA_only_neg_emotion_density"
    ]["boundary_warning"],
)

print(
    "Estimator-sensitive:",
    family1_separate_model_info[
        "NA_only_neg_emotion_density"
    ]["estimator_sensitive"],
)

In [ ]:
# ============================================================
# ADDITIONAL FAMILY 1 — DIAGNOSE PA-ONLY NEGATIVE-EMOTION MODEL
# ============================================================

pa_neg_emotion_formula = (
    "neg_emotion_density ~ "
    "mood_PA_within_z + "
    "mood_PA_between_z + "
    "gender_female + "
    "family_ses_rev_z + "
    "nwords_z"
)

pa_neg_emotion_candidates = {}
pa_neg_emotion_diagnostics = []

for optimizer in [
    "bfgs",
    "lbfgs",
    "cg",
    "powell",
    "nm",
]:
    try:
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")

            model = smf.mixedlm(
                formula=pa_neg_emotion_formula,
                data=family1_separate_data,
                groups=family1_separate_data[
                    "participantid"
                ],
            ).fit(
                reml=False,
                method=optimizer,
                disp=False,
            )

        warning_messages = [
            str(item.message)
            for item in caught
        ]

        pa_neg_emotion_candidates[
            optimizer
        ] = model

        pa_neg_emotion_diagnostics.append({
            "optimizer": optimizer,
            "converged": bool(model.converged),
            "intercept_variance": float(
                model.cov_re.iloc[0, 0]
            ),
            "residual_variance": float(
                model.scale
            ),
            "log_likelihood": float(
                model.llf
            ),
            "warnings": " | ".join(
                warning_messages
            ),
            "error": "",
        })

    except Exception as error:
        pa_neg_emotion_diagnostics.append({
            "optimizer": optimizer,
            "converged": False,
            "intercept_variance": np.nan,
            "residual_variance": np.nan,
            "log_likelihood": np.nan,
            "warnings": "",
            "error": (
                f"{type(error).__name__}: {error}"
            ),
        })

pa_neg_emotion_diagnostics = pd.DataFrame(
    pa_neg_emotion_diagnostics
)

display(pa_neg_emotion_diagnostics)

In [ ]:
pa_neg_emotion_powell = (
    pa_neg_emotion_candidates["powell"]
)

pa_neg_emotion_intercept_var = float(
    pa_neg_emotion_powell.cov_re.iloc[0, 0]
)

pa_neg_emotion_residual_var = float(
    pa_neg_emotion_powell.scale
)

pa_neg_emotion_icc = (
    pa_neg_emotion_intercept_var
    / (
        pa_neg_emotion_intercept_var
        + pa_neg_emotion_residual_var
    )
)

print("Model ID: PA_only_neg_emotion_density")
print("Optimizer: powell")
print("Converged:", pa_neg_emotion_powell.converged)
print("Observations:", int(pa_neg_emotion_powell.nobs))
print(
    "Participants:",
    int(pa_neg_emotion_powell.model.n_groups),
)
print(
    "Participant intercept variance:",
    pa_neg_emotion_intercept_var,
)
print(
    "Residual variance:",
    pa_neg_emotion_residual_var,
)
print("ICC:", pa_neg_emotion_icc)
print(
    "Log-likelihood:",
    float(pa_neg_emotion_powell.llf),
)

for term in [
    "mood_PA_within_z",
    "mood_PA_between_z",
]:
    print(
        term,
        "beta =",
        float(pa_neg_emotion_powell.fe_params[term]),
        "SE =",
        float(pa_neg_emotion_powell.bse[term]),
        "p =",
        float(pa_neg_emotion_powell.pvalues[term]),
    )

In [ ]:
pa_neg_emotion_cluster = smf.ols(
    formula=pa_neg_emotion_formula,
    data=family1_separate_data,
).fit(
    cov_type="cluster",
    cov_kwds={
        "groups": family1_separate_data["participantid"]
    },
)

print(
    "Clustered OLS observations:",
    int(pa_neg_emotion_cluster.nobs),
)

for term in [
    "mood_PA_within_z",
    "mood_PA_between_z",
]:
    print(
        term,
        "beta =",
        float(pa_neg_emotion_cluster.params[term]),
        "SE =",
        float(pa_neg_emotion_cluster.bse[term]),
        "p =",
        float(pa_neg_emotion_cluster.pvalues[term]),
    )

In [ ]:
family1_separate_models[
    "PA_only_neg_emotion_density"
] = pa_neg_emotion_powell

family1_separate_model_info[
    "PA_only_neg_emotion_density"
] = {
    "optimizer": "powell",
    "warnings": [
        "The MLE may be on the boundary "
        "of the parameter space."
    ],
    "boundary_warning": True,
    "estimator_sensitive": True,
    "participant_intercept_variance": float(
        pa_neg_emotion_powell.cov_re.iloc[0, 0]
    ),
    "residual_variance": float(
        pa_neg_emotion_powell.scale
    ),
    "ICC": float(pa_neg_emotion_icc),
}

family1_separate_cluster_models[
    "PA_only_neg_emotion_density"
] = pa_neg_emotion_cluster

print(
    "Number of stored mixed models:",
    len(family1_separate_models),
)

print(
    "Number of stored clustered models:",
    len(family1_separate_cluster_models),
)

print(
    "Optimizer:",
    family1_separate_model_info[
        "PA_only_neg_emotion_density"
    ]["optimizer"],
)

print(
    "Boundary warning:",
    family1_separate_model_info[
        "PA_only_neg_emotion_density"
    ]["boundary_warning"],
)

print(
    "Estimator-sensitive:",
    family1_separate_model_info[
        "PA_only_neg_emotion_density"
    ]["estimator_sensitive"],
)

In [ ]:
# ============================================================
# ADDITIONAL FAMILY 1 — AUDIT ALL STORED MODELS
# ============================================================

expected_model_ids = [
    "NA_only_social_conn_llm",
    "NA_only_sent_score",
    "NA_only_social_density",
    "NA_only_neg_emotion_density",
    "PA_only_social_conn_llm",
    "PA_only_sent_score",
    "PA_only_social_density",
    "PA_only_neg_emotion_density",
]

mixed_ids = set(family1_separate_models.keys())
cluster_ids = set(family1_separate_cluster_models.keys())
info_ids = set(family1_separate_model_info.keys())
expected_ids = set(expected_model_ids)

print("Mixed models stored:", len(mixed_ids))
print("Clustered models stored:", len(cluster_ids))
print("Model-info records stored:", len(info_ids))

print(
    "Missing mixed models:",
    sorted(expected_ids - mixed_ids),
)

print(
    "Missing clustered models:",
    sorted(expected_ids - cluster_ids),
)

print(
    "Missing model-info records:",
    sorted(expected_ids - info_ids),
)

audit_rows = []

for model_id in expected_model_ids:
    mixed_model = family1_separate_models[model_id]
    cluster_model = family1_separate_cluster_models[model_id]
    model_info = family1_separate_model_info[model_id]

    audit_rows.append({
        "model_id": model_id,
        "mixed_converged": bool(mixed_model.converged),
        "mixed_N": int(mixed_model.nobs),
        "mixed_participants": int(
            mixed_model.model.n_groups
        ),
        "cluster_N": int(cluster_model.nobs),
        "optimizer": model_info["optimizer"],
        "boundary_warning": model_info.get(
            "boundary_warning",
            False,
        ),
        "estimator_sensitive": model_info.get(
            "estimator_sensitive",
            False,
        ),
    })

family1_separate_audit = pd.DataFrame(audit_rows)

display(family1_separate_audit)

assert mixed_ids == expected_ids
assert cluster_ids == expected_ids
assert info_ids == expected_ids

assert family1_separate_audit[
    "mixed_converged"
].all()

assert (
    family1_separate_audit["mixed_N"] == 429
).all()

assert (
    family1_separate_audit["mixed_participants"] == 40
).all()

assert (
    family1_separate_audit["cluster_N"] == 429
).all()

print(
    "\nAudit passed: all eight mixed models and "
    "all eight clustered models are complete."
)

In [ ]:
# ============================================================
# ADDITIONAL FAMILY 1 — CONSOLIDATE FOCAL RESULTS
# ============================================================

model_definitions = {
    "NA_only_social_conn_llm": {
        "specification": "Negative mood only",
        "outcome": "LLM social connectedness",
        "within_term": "mood_NA_within_z",
        "between_term": "mood_NA_between_z",
    },
    "NA_only_sent_score": {
        "specification": "Negative mood only",
        "outcome": "Sentiment score",
        "within_term": "mood_NA_within_z",
        "between_term": "mood_NA_between_z",
    },
    "NA_only_social_density": {
        "specification": "Negative mood only",
        "outcome": "Social-word density",
        "within_term": "mood_NA_within_z",
        "between_term": "mood_NA_between_z",
    },
    "NA_only_neg_emotion_density": {
        "specification": "Negative mood only",
        "outcome": "Negative-emotion density",
        "within_term": "mood_NA_within_z",
        "between_term": "mood_NA_between_z",
    },
    "PA_only_social_conn_llm": {
        "specification": "Positive mood only",
        "outcome": "LLM social connectedness",
        "within_term": "mood_PA_within_z",
        "between_term": "mood_PA_between_z",
    },
    "PA_only_sent_score": {
        "specification": "Positive mood only",
        "outcome": "Sentiment score",
        "within_term": "mood_PA_within_z",
        "between_term": "mood_PA_between_z",
    },
    "PA_only_social_density": {
        "specification": "Positive mood only",
        "outcome": "Social-word density",
        "within_term": "mood_PA_within_z",
        "between_term": "mood_PA_between_z",
    },
    "PA_only_neg_emotion_density": {
        "specification": "Positive mood only",
        "outcome": "Negative-emotion density",
        "within_term": "mood_PA_within_z",
        "between_term": "mood_PA_between_z",
    },
}

consolidated_rows = []

for model_id, definition in model_definitions.items():

    mixed_model = family1_separate_models[model_id]

    clustered_model = (
        family1_separate_cluster_models[model_id]
    )

    model_info = family1_separate_model_info[model_id]

    participant_variance = float(
        mixed_model.cov_re.iloc[0, 0]
    )
    residual_variance = float(
        mixed_model.scale
    )
    icc = float(
        model_info.get(
            "ICC",
            participant_variance
            / (participant_variance + residual_variance),
        )
    )

    for component, term in [
        ("Within-person", definition["within_term"]),
        ("Between-person", definition["between_term"]),
    ]:
        consolidated_rows.append({
            "model_id": model_id,
            "specification": definition["specification"],
            "outcome": definition["outcome"],
            "component": component,
            "term": term,
            "mixed_beta": float(
                mixed_model.fe_params[term]
            ),
            "mixed_SE": float(
                mixed_model.bse[term]
            ),
            "mixed_p": float(
                mixed_model.pvalues[term]
            ),
            "cluster_beta": float(
                clustered_model.params[term]
            ),
            "cluster_SE": float(
                clustered_model.bse[term]
            ),
            "cluster_p": float(
                clustered_model.pvalues[term]
            ),
            "N_observations": int(
                mixed_model.nobs
            ),
            "N_participants": int(
                mixed_model.model.n_groups
            ),
            "optimizer": model_info.get("optimizer", ""),
            "participant_intercept_variance": (
                participant_variance
            ),
            "residual_variance": residual_variance,
            "ICC": icc,
            "boundary_warning": model_info.get(
                "boundary_warning",
                False,
            ),
            "estimator_sensitive": model_info.get(
                "estimator_sensitive",
                False,
            ),
        })

family1_separate_consolidated = pd.DataFrame(
    consolidated_rows
)

display(
    family1_separate_consolidated.round({
        "mixed_beta": 6,
        "mixed_SE": 6,
        "mixed_p": 6,
        "cluster_beta": 6,
        "cluster_SE": 6,
        "cluster_p": 6,
        "participant_intercept_variance": 8,
        "residual_variance": 8,
        "ICC": 4,
    })
)

print(
    "Consolidated rows:",
    len(family1_separate_consolidated),
)

print(
    "Unique models:",
    family1_separate_consolidated[
        "model_id"
    ].nunique(),
)

assert len(family1_separate_consolidated) == 16

assert (
    family1_separate_consolidated[
        "model_id"
    ].nunique()
    == 8
)

print(
    "\nConsolidation passed: "
    "16 focal estimates from 8 models."
)

In [ ]:
# Save consolidated separate Family 1 results

family1_separate_results_path = (
    TABLES
    / "Family1_separate_mood_models_consolidated_common429.csv"
)

family1_separate_consolidated.to_csv(
    family1_separate_results_path,
    index=False,
    encoding="utf-8-sig",
)

print("Saved:", family1_separate_results_path)
print("File exists:", family1_separate_results_path.exists())
print("Rows saved:", len(family1_separate_consolidated))
print(
    "Models saved:",
    family1_separate_consolidated["model_id"].nunique(),
)

In [ ]:
family1_separate_audit_path = (
    TABLES
    / "Family1_separate_model_health_common429.csv"
)

family1_separate_audit.to_csv(
    family1_separate_audit_path,
    index=False,
    encoding="utf-8-sig",
)

print("Saved:", family1_separate_audit_path)
print("File exists:", family1_separate_audit_path.exists())
print("Rows saved:", len(family1_separate_audit))

print(
    "Converged models:",
    int(
        family1_separate_audit[
            "mixed_converged"
        ].sum()
    ),
)

print(
    "Models with boundary warnings:",
    int(
        family1_separate_audit[
            "boundary_warning"
        ].sum()
    ),
)

print(
    "Estimator-sensitive models:",
    int(
        family1_separate_audit[
            "estimator_sensitive"
        ].sum()
    ),
)

In [ ]:
# ============================================================
# ADDITIONAL FAMILY 1 — COMPLETE MIXED-MODEL RESULTS
# ============================================================

full_result_rows = []

for model_id in expected_model_ids:

    model = family1_separate_models[model_id]
    model_info = family1_separate_model_info[model_id]
    definition = model_definitions[model_id]

    for term in model.fe_params.index:

        participant_intercept_variance = float(
            model.cov_re.iloc[0, 0]
        )
        residual_variance = float(model.scale)
        icc = float(
            model_info.get(
            "ICC",
            participant_intercept_variance
            / (
                participant_intercept_variance
                + residual_variance
            ),
            )
        )

        full_result_rows.append({
            "model_id": model_id,
            "specification": definition["specification"],
            "outcome": definition["outcome"],
            "term": term,
            "beta": float(model.fe_params[term]),
            "SE": float(model.bse[term]),
            "p": float(model.pvalues[term]),
            "N_observations": int(model.nobs),
            "N_participants": int(model.model.n_groups),
            "optimizer": model_info["optimizer"],
            "participant_intercept_variance": participant_intercept_variance,
            "residual_variance": residual_variance,
            "ICC": icc,
            "boundary_warning": model_info.get("boundary_warning", False),
            "estimator_sensitive": model_info.get("estimator_sensitive", False),
        })

family1_separate_full_results = pd.DataFrame(
    full_result_rows
)

print(
    "Rows created:",
    len(family1_separate_full_results),
)

print(
    "Models represented:",
    family1_separate_full_results[
        "model_id"
    ].nunique(),
)

print(
    "Terms per model:"
)

print(
    family1_separate_full_results.groupby(
        "model_id"
    )["term"].count()
)

assert (
    family1_separate_full_results[
        "model_id"
    ].nunique()
    == 8
)

assert (
    family1_separate_full_results.groupby(
        "model_id"
    )["term"].count()
    == 6
).all()

print(
    "\nFull-results construction passed: "
    "6 fixed effects for each of 8 models."
)

In [ ]:
# ============================================================
# ADDITIONAL FAMILY 1 — SAVE COMPLETE MIXED-MODEL RESULTS
# ============================================================

family1_separate_full_path = (
    TABLES
    / "Family1_separate_mood_models_full_common429.csv"
)

family1_separate_full_results.to_csv(
    family1_separate_full_path,
    index=False,
    encoding="utf-8-sig",
)

print("Saved:", family1_separate_full_path)
print("File exists:", family1_separate_full_path.exists())
print("Rows saved:", len(family1_separate_full_results))

print(
    "Models saved:",
    family1_separate_full_results[
        "model_id"
    ].nunique(),
)

print(
    "Terms per model:",
    family1_separate_full_results.groupby(
        "model_id"
    )["term"].count().unique().tolist(),
)

In [ ]:
# ============================================================
# ADDITIONAL FAMILY 1 — LOAD ORIGINAL COMBINED MODEL RESULTS
# ============================================================

family1_combined_path = (
    TABLES
    / "Family1_mixed_models_common429.csv"
)

print("Combined-results path:", family1_combined_path)
print("File exists:", family1_combined_path.exists())

assert family1_combined_path.exists(), (
    "The original combined Family 1 results file "
    "was not found."
)

family1_combined_results = pd.read_csv(
    family1_combined_path
)

print(
    "Rows loaded:",
    len(family1_combined_results),
)

print(
    "Outcomes loaded:",
    family1_combined_results[
        "outcome"
    ].nunique(),
)

print(
    "Columns:",
    family1_combined_results.columns.tolist(),
)

display(
    family1_combined_results.head(10)
)

In [ ]:
# ============================================================
# ADDITIONAL FAMILY 1 — COMPARE COMBINED AND SEPARATE MODELS
# ============================================================

outcome_name_map = {
    "social_conn_llm": "LLM social connectedness",
    "sent_score": "Sentiment score",
    "social_density": "Social-word density",
    "neg_emotion_density": "Negative-emotion density",
}

comparison_rows = []

for model_id, definition in model_definitions.items():

    separate_model = family1_separate_models[model_id]
    clustered_model = family1_separate_cluster_models[model_id]

    if model_id.startswith("NA_only_"):
        mood = "Negative mood"
        outcome_code = model_id.replace("NA_only_", "")
        terms = [
            ("Within-person", "mood_NA_within_z"),
            ("Between-person", "mood_NA_between_z"),
        ]
    else:
        mood = "Positive mood"
        outcome_code = model_id.replace("PA_only_", "")
        terms = [
            ("Within-person", "mood_PA_within_z"),
            ("Between-person", "mood_PA_between_z"),
        ]

    for component, term in terms:

        combined_row = family1_combined_results.loc[
            (
                family1_combined_results["outcome"]
                == outcome_code
            )
            & (
                family1_combined_results["term"]
                == term
            )
        ]

        assert len(combined_row) == 1, (
            f"Expected one combined-model row for "
            f"{outcome_code}, {term}; found "
            f"{len(combined_row)}."
        )

        combined_row = combined_row.iloc[0]

        combined_supported = (
            float(combined_row["p"]) < 0.05
        )

        separate_supported = (
            float(separate_model.pvalues[term]) < 0.05
        )

        clustered_supported = (
            float(clustered_model.pvalues[term]) < 0.05
        )

        comparison_rows.append({
            "model_id": model_id,
            "mood": mood,
            "outcome": outcome_name_map[outcome_code],
            "component": component,
            "term": term,
            "combined_beta": float(
                combined_row["beta"]
            ),
            "combined_SE": float(
                combined_row["SE"]
            ),
            "combined_p": float(
                combined_row["p"]
            ),
            "separate_beta": float(
                separate_model.fe_params[term]
            ),
            "separate_SE": float(
                separate_model.bse[term]
            ),
            "separate_p": float(
                separate_model.pvalues[term]
            ),
            "cluster_beta": float(
                clustered_model.params[term]
            ),
            "cluster_SE": float(
                clustered_model.bse[term]
            ),
            "cluster_p": float(
                clustered_model.pvalues[term]
            ),
            "combined_supported": combined_supported,
            "separate_supported": separate_supported,
            "clustered_supported": clustered_supported,
            "conclusion_changed": (
                combined_supported
                != separate_supported
            ),
            "separate_robust_to_clustering": (
                separate_supported
                == clustered_supported
            ),
        })

family1_combined_vs_separate = pd.DataFrame(
    comparison_rows
)

display(
    family1_combined_vs_separate.round({
        "combined_beta": 6,
        "combined_SE": 6,
        "combined_p": 6,
        "separate_beta": 6,
        "separate_SE": 6,
        "separate_p": 6,
        "cluster_beta": 6,
        "cluster_SE": 6,
        "cluster_p": 6,
    })
)

print(
    "Comparison rows:",
    len(family1_combined_vs_separate),
)

print(
    "Combined-to-separate conclusion changes:",
    int(
        family1_combined_vs_separate[
            "conclusion_changed"
        ].sum()
    ),
)

print(
    "Separate conclusions robust to clustering:",
    int(
        family1_combined_vs_separate[
            "separate_robust_to_clustering"
        ].sum()
    ),
    "of",
    len(family1_combined_vs_separate),
)

assert len(family1_combined_vs_separate) == 16

print(
    "\nComparison passed: all 16 mood estimates matched."
)

In [ ]:
family1_comparison_path = (
    TABLES
    / "Family1_combined_vs_separate_comparison_common429.csv"
)

family1_combined_vs_separate.to_csv(
    family1_comparison_path,
    index=False,
    encoding="utf-8-sig",
)

print("Saved:", family1_comparison_path)
print("File exists:", family1_comparison_path.exists())
print("Rows saved:", len(family1_combined_vs_separate))

print(
    "Conclusion changes:",
    int(
        family1_combined_vs_separate[
            "conclusion_changed"
        ].sum()
    ),
)

print(
    "Robust to clustering:",
    int(
        family1_combined_vs_separate[
            "separate_robust_to_clustering"
        ].sum()
    ),
    "of",
    len(family1_combined_vs_separate),
)

In [ ]:
# ============================================================
# ADDITIONAL FAMILY 1 — DEFINE THESIS TABLE FORMATTING
# ============================================================

family1_separate_outcome_labels = {
    "social_conn_llm": "LLM social connectedness",
    "sent_score": "Sentiment score",
    "social_density": "Social-word density",
    "neg_emotion_density": "Negative-emotion density",
}

family1_separate_term_labels = {
    "Intercept": "Intercept",
    "mood_NA_within_z": "Negative mood, within-person",
    "mood_NA_between_z": "Negative mood, between-person",
    "mood_PA_within_z": "Positive mood, within-person",
    "mood_PA_between_z": "Positive mood, between-person",
    "gender_female": "Female indicator",
    "family_ses_rev_z": "Socioeconomic position",
    "nwords_z": "Diary word count",
}


def format_family1_separate_cell(model, term):
    beta = float(model.fe_params[term])
    standard_error = float(model.bse[term])
    p_value = float(model.pvalues[term])

    if p_value < 0.001:
        significance = "***"
    elif p_value < 0.01:
        significance = "**"
    elif p_value < 0.05:
        significance = "*"
    else:
        significance = ""

    return (
        f"{beta:.3f}{significance} "
        f"({standard_error:.3f})"
    )


print("Outcome labels:", len(family1_separate_outcome_labels))
print("Term labels:", len(family1_separate_term_labels))

print(
    "Example:",
    format_family1_separate_cell(
        family1_separate_models[
            "NA_only_social_conn_llm"
        ],
        "mood_NA_within_z",
    ),
)

assert len(family1_separate_outcome_labels) == 4
assert len(family1_separate_term_labels) == 8

print(
    "\nThesis-table labels and formatter "
    "defined successfully."
)

In [ ]:
# ============================================================
# ADDITIONAL FAMILY 1 — DEFINE THESIS TABLE FORMATTING
# ============================================================

family1_separate_outcome_labels = {
    "social_conn_llm": "LLM social connectedness",
    "sent_score": "Sentiment score",
    "social_density": "Social-word density",
    "neg_emotion_density": "Negative-emotion density",
}

family1_separate_term_labels = {
    "Intercept": "Intercept",
    "mood_NA_within_z": "Negative mood, within-person",
    "mood_NA_between_z": "Negative mood, between-person",
    "mood_PA_within_z": "Positive mood, within-person",
    "mood_PA_between_z": "Positive mood, between-person",
    "gender_female": "Female indicator",
    "family_ses_rev_z": "Socioeconomic position",
    "nwords_z": "Diary word count",
}


def format_family1_separate_cell(model, term):
    beta = float(model.fe_params[term])
    standard_error = float(model.bse[term])
    p_value = float(model.pvalues[term])

    if p_value < 0.001:
        significance = "***"
    elif p_value < 0.01:
        significance = "**"
    elif p_value < 0.05:
        significance = "*"
    else:
        significance = ""

    return (
        f"{beta:.3f}{significance} "
        f"({standard_error:.3f})"
    )


print("Outcome labels:", len(family1_separate_outcome_labels))
print("Term labels:", len(family1_separate_term_labels))

print(
    "Example:",
    format_family1_separate_cell(
        family1_separate_models[
            "NA_only_social_conn_llm"
        ],
        "mood_NA_within_z",
    ),
)

assert len(family1_separate_outcome_labels) == 4
assert len(family1_separate_term_labels) == 8

print(
    "\nThesis-table labels and formatter "
    "defined successfully."
)

In [ ]:
# ============================================================
# ADDITIONAL FAMILY 1 — BUILD TABLE 3, PANEL A
# NEGATIVE-MOOD-ONLY MODELS
# ============================================================

panel_a_outcomes = [
    "social_conn_llm",
    "sent_score",
    "social_density",
    "neg_emotion_density",
]

panel_a_terms = [
    "Intercept",
    "mood_NA_within_z",
    "mood_NA_between_z",
    "gender_female",
    "family_ses_rev_z",
    "nwords_z",
]

panel_a_rows = []

for term in panel_a_terms:

    row = {
        "Predictor": family1_separate_term_labels[term]
    }

    for outcome in panel_a_outcomes:

        model_id = f"NA_only_{outcome}"
        model = family1_separate_models[model_id]

        outcome_label = (
            family1_separate_outcome_labels[outcome]
        )

        row[outcome_label] = (
            format_family1_separate_cell(
                model,
                term,
            )
        )

    panel_a_rows.append(row)

model_information = [
    (
        "N observations",
        lambda model: str(int(model.nobs)),
    ),
    (
        "N participants",
        lambda model: str(
            int(model.model.n_groups)
        ),
    ),
    (
        "Participant intercept variance",
        lambda model: (
            f"{float(model.cov_re.iloc[0, 0]):.6f}"
        ),
    ),
    (
        "Residual variance",
        lambda model: (
            f"{float(model.scale):.6f}"
        ),
    ),
    (
        "ICC",
        lambda model: f"{float(model.cov_re.iloc[0, 0]) / (float(model.cov_re.iloc[0, 0]) + float(model.scale)):.3f}",
    ),
]

for information_label, value_function in model_information:

    row = {
        "Predictor": information_label
    }

    for outcome in panel_a_outcomes:

        model_id = f"NA_only_{outcome}"
        model = family1_separate_models[model_id]

        outcome_label = (
            family1_separate_outcome_labels[outcome]
        )

        row[outcome_label] = value_function(model)

    panel_a_rows.append(row)

table3_panel_a = pd.DataFrame(panel_a_rows)

display(table3_panel_a)

print("Panel A rows:", len(table3_panel_a))
print("Panel A columns:", len(table3_panel_a.columns))

assert len(table3_panel_a) == 11
assert len(table3_panel_a.columns) == 5

print("\nPanel A construction passed.")

In [ ]:
# ============================================================
# ADDITIONAL FAMILY 1 — BUILD TABLE 3, PANEL B
# POSITIVE-MOOD-ONLY MODELS
# ============================================================

panel_b_outcomes = [
    "social_conn_llm",
    "sent_score",
    "social_density",
    "neg_emotion_density",
]

panel_b_terms = [
    "Intercept",
    "mood_PA_within_z",
    "mood_PA_between_z",
    "gender_female",
    "family_ses_rev_z",
    "nwords_z",
]

panel_b_rows = []

for term in panel_b_terms:

    row = {
        "Predictor": family1_separate_term_labels[term]
    }

    for outcome in panel_b_outcomes:

        model_id = f"PA_only_{outcome}"
        model = family1_separate_models[model_id]

        outcome_label = (
            family1_separate_outcome_labels[outcome]
        )

        row[outcome_label] = (
            format_family1_separate_cell(
                model,
                term,
            )
        )

    panel_b_rows.append(row)

for information_label, value_function in model_information:

    row = {
        "Predictor": information_label
    }

    for outcome in panel_b_outcomes:

        model_id = f"PA_only_{outcome}"
        model = family1_separate_models[model_id]

        outcome_label = (
            family1_separate_outcome_labels[outcome]
        )

        row[outcome_label] = value_function(model)

    panel_b_rows.append(row)

table3_panel_b = pd.DataFrame(panel_b_rows)

display(table3_panel_b)

print("Panel B rows:", len(table3_panel_b))
print("Panel B columns:", len(table3_panel_b.columns))

assert len(table3_panel_b) == 11
assert len(table3_panel_b.columns) == 5

print("\nPanel B construction passed.")

In [ ]:
# ============================================================
# ADDITIONAL FAMILY 1 — SAVE TABLE 3 PANELS
# ============================================================

table3_panel_a_path = (
    TABLES
    / "Table_3a_Family1_NA_only_common429.csv"
)

table3_panel_b_path = (
    TABLES
    / "Table_3b_Family1_PA_only_common429.csv"
)

table3_panel_a.to_csv(
    table3_panel_a_path,
    index=False,
    encoding="utf-8-sig",
)

table3_panel_b.to_csv(
    table3_panel_b_path,
    index=False,
    encoding="utf-8-sig",
)

print("Panel A saved:", table3_panel_a_path)
print("Panel A exists:", table3_panel_a_path.exists())
print("Panel A rows:", len(table3_panel_a))

print()

print("Panel B saved:", table3_panel_b_path)
print("Panel B exists:", table3_panel_b_path.exists())
print("Panel B rows:", len(table3_panel_b))

In [ ]:
# ============================================================
# ADDITIONAL FAMILY 1 — COMBINE TABLE 3 PANELS
# ============================================================

table_columns = table3_panel_a.columns.tolist()

panel_a_heading = pd.DataFrame([{
    table_columns[0]: (
        "Panel A. Negative-mood-only models"
    ),
    **{
        column: ""
        for column in table_columns[1:]
    },
}])

panel_b_heading = pd.DataFrame([{
    table_columns[0]: (
        "Panel B. Positive-mood-only models"
    ),
    **{
        column: ""
        for column in table_columns[1:]
    },
}])

blank_row = pd.DataFrame([{
    column: ""
    for column in table_columns
}])

table3_separate_final = pd.concat(
    [
        panel_a_heading,
        table3_panel_a,
        blank_row,
        panel_b_heading,
        table3_panel_b,
    ],
    ignore_index=True,
)

display(table3_separate_final)

print(
    "Combined Table 3 rows:",
    len(table3_separate_final),
)

print(
    "Combined Table 3 columns:",
    len(table3_separate_final.columns),
)

assert len(table3_separate_final) == 25
assert len(table3_separate_final.columns) == 5

print(
    "\nCombined two-panel Table 3 "
    "constructed successfully."
)

In [ ]:
table3_separate_path = (
    TABLES
    / "Table_3_Family1_separate_mood_models_common429.csv"
)

table3_separate_final.to_csv(
    table3_separate_path,
    index=False,
    encoding="utf-8-sig",
)

print("Saved:", table3_separate_path)
print("File exists:", table3_separate_path.exists())
print("Rows saved:", len(table3_separate_final))
print("Columns saved:", len(table3_separate_final.columns))

In [ ]:
required_family2_objects = [
    "family2_data",
    "family2_models",
    "n_friends_z_person",
]

for object_name in required_family2_objects:
    print(
        object_name,
        "available:",
        object_name in globals(),
    )

print(
    "Corrected friendship column in analysis:",
    (
        "n_friends_z_person"
        in analysis.columns
    ),
)

In [ ]:
# ============================================================
# FINAL MODELS 7 — FIT FAMILY 2 MODERATION MODELS
# ============================================================

import warnings
import statsmodels.formula.api as smf

family2_data = analysis[
    analysis["common_language_sample"]
].copy()

family2_outcomes = [
    "social_conn_llm",
    "sent_score",
    "social_density",
    "neg_emotion_density",
]

family2_mood_families = {
    "NA": {
        "within": "mood_NA_within_z",
        "between": "mood_NA_between_z",
    },
    "PA": {
        "within": "mood_PA_within_z",
        "between": "mood_PA_between_z",
    },
}

family2_controls = [
    "gender_female",
    "family_ses_rev_z",
    "nwords_z",
]

required_family2 = [
    "participantid",
    "n_friends_z_person",
] + family2_outcomes + family2_controls + [
    "mood_NA_within_z",
    "mood_NA_between_z",
    "mood_PA_within_z",
    "mood_PA_between_z",
]

print("Family 2 rows:", len(family2_data))
print(
    "Family 2 participants:",
    family2_data["participantid"].nunique()
)
print(
    "Missing values across required variables:",
    int(family2_data[required_family2].isna().sum().sum())
)

family2_models = {}
family2_health_rows = []

optimizers = [
    "lbfgs",
    "bfgs",
    "cg",
    "powell",
    "nm",
]

for mood_family, mood_terms in family2_mood_families.items():

    mood_within = mood_terms["within"]
    mood_between = mood_terms["between"]

    for outcome in family2_outcomes:

        model_id = f"{mood_family}_{outcome}"

        formula = (
            f"{outcome} ~ "
            f"{mood_within} * n_friends_z_person + "
            f"{mood_between} * n_friends_z_person + "
            "gender_female + family_ses_rev_z + nwords_z"
        )

        fitted_model = None
        optimizer_used = None
        selected_warning_messages = []
        last_error = None

        for optimizer in optimizers:

            try:
                with warnings.catch_warnings(record=True) as caught:
                    warnings.simplefilter("always")

                    candidate = smf.mixedlm(
                        formula=formula,
                        data=family2_data,
                        groups=family2_data["participantid"],
                    ).fit(
                        reml=True,
                        method=optimizer,
                        maxiter=2000,
                        disp=False,
                    )

                # Retain warnings only from the selected converged fit
                current_warnings = [
                    str(w.message)
                    for w in caught
                ]

                if candidate.converged:
                    fitted_model = candidate
                    optimizer_used = optimizer
                    selected_warning_messages = current_warnings
                    break

            except Exception as error:
                last_error = str(error)

        if fitted_model is not None:

            participant_variance = float(
                fitted_model.cov_re.iloc[0, 0]
            )

            residual_variance = float(
                fitted_model.scale
            )

            icc = (
                participant_variance
                / (participant_variance + residual_variance)
            )

            family2_models[model_id] = fitted_model

            family2_health_rows.append({
                "model_id": model_id,
                "mood_family": mood_family,
                "outcome": outcome,
                "N_observations": int(fitted_model.nobs),
                "N_participants":
                    int(fitted_model.model.n_groups),
                "optimizer": optimizer_used,
                "converged": fitted_model.converged,
                "participant_intercept_variance":
                    participant_variance,
                "residual_variance":
                    residual_variance,
                "ICC": icc,
                "boundary_or_singular_warning": any(
                    "boundary" in message.lower()
                    or "singular" in message.lower()
                    for message in selected_warning_messages
                ),
                "warning_text": " | ".join(
                    selected_warning_messages
                ),
                "last_error": "",
            })

        else:

            family2_health_rows.append({
                "model_id": model_id,
                "mood_family": mood_family,
                "outcome": outcome,
                "N_observations": len(family2_data),
                "N_participants":
                    family2_data["participantid"].nunique(),
                "optimizer": None,
                "converged": False,
                "participant_intercept_variance": np.nan,
                "residual_variance": np.nan,
                "ICC": np.nan,
                "boundary_or_singular_warning": None,
                "warning_text": "",
                "last_error": last_error,
            })

family2_health = pd.DataFrame(
    family2_health_rows
)

family2_health_display = family2_health[
    [
        "model_id",
        "N_observations",
        "N_participants",
        "optimizer",
        "converged",
        "participant_intercept_variance",
        "residual_variance",
        "ICC",
        "boundary_or_singular_warning",
    ]
].copy()

family2_health_display[
    [
        "participant_intercept_variance",
        "residual_variance",
        "ICC",
    ]
] = family2_health_display[
    [
        "participant_intercept_variance",
        "residual_variance",
        "ICC",
    ]
].round(6)

print("\nFamily 2 model-health check:")
display(family2_health_display)

In [ ]:
# Verify the two sentiment interaction models

sentiment_interactions = {
    "NA_sent_score": (
        "mood_NA_within_z:n_friends_z_person"
    ),
    "PA_sent_score": (
        "mood_PA_within_z:n_friends_z_person"
    ),
}

for model_id, interaction_term in sentiment_interactions.items():

    model = family2_models[model_id]

    print("Model:", model_id)
    print("Term:", interaction_term)
    print(
        "Beta:",
        float(model.params[interaction_term]),
    )
    print(
        "SE:",
        float(model.bse[interaction_term]),
    )
    print(
        "p:",
        float(model.pvalues[interaction_term]),
    )
    print(
        "Converged:",
        bool(model.converged),
    )
    print("-" * 50)

In [ ]:
# ============================================================
# SUPPLEMENTARY FIGURE — NA WITHIN × FRIENDS → SENTIMENT
# CREATE PREDICTION DATA
# ============================================================

na_sentiment_model = family2_models["NA_sent_score"]

friend_mean = friends_mean_person
friend_sd = friends_sd_person

friend_levels_raw = {
    "Low: 0 friends": 0.0,
    f"Average: {friend_mean:.1f} friends": friend_mean,
    f"Higher: {friend_mean + friend_sd:.1f} friends": (
        friend_mean + friend_sd
    ),
}

mood_grid = np.linspace(
    family2_data["mood_NA_within_z"].min(),
    family2_data["mood_NA_within_z"].max(),
    100,
)

fixed_effect_names = (
    na_sentiment_model.fe_params.index.tolist()
)

fixed_covariance = (
    na_sentiment_model.cov_params()
    .loc[fixed_effect_names, fixed_effect_names]
)

prediction_rows = []

for friend_label, raw_friend_value in friend_levels_raw.items():

    friend_z = (
        raw_friend_value - friend_mean
    ) / friend_sd

    for mood_value in mood_grid:

        predictor_values = {
            "Intercept": 1.0,
            "mood_NA_within_z": mood_value,
            "mood_NA_between_z": 0.0,
            "n_friends_z_person": friend_z,
            (
                "mood_NA_within_z:"
                "n_friends_z_person"
            ): mood_value * friend_z,
            (
                "mood_NA_between_z:"
                "n_friends_z_person"
            ): 0.0,
            "gender_female": 0.0,
            "family_ses_rev_z": 0.0,
            "nwords_z": 0.0,
        }

        design_vector = np.array([
            predictor_values[name]
            for name in fixed_effect_names
        ])

        beta = (
            na_sentiment_model.fe_params
            .loc[fixed_effect_names]
            .to_numpy()
        )

        predicted_value = float(
            design_vector @ beta
        )

        prediction_variance = float(
            design_vector
            @ fixed_covariance.to_numpy()
            @ design_vector
        )

        prediction_se = np.sqrt(
            max(prediction_variance, 0.0)
        )

        prediction_rows.append({
            "Negative mood within-person (z)": mood_value,
            "Predicted sentiment": predicted_value,
            "CI lower": (
                predicted_value
                - 1.96 * prediction_se
            ),
            "CI upper": (
                predicted_value
                + 1.96 * prediction_se
            ),
            "Friendship network": friend_label,
            "Raw friend count": raw_friend_value,
            "Friend count z": friend_z,
        })

na_sentiment_predictions = pd.DataFrame(
    prediction_rows
)

print(
    "Prediction rows:",
    len(na_sentiment_predictions),
)

print(
    "Friendship levels:",
    na_sentiment_predictions[
        "Friendship network"
    ].unique().tolist(),
)

print(
    "Missing prediction values:",
    int(
        na_sentiment_predictions[
            [
                "Predicted sentiment",
                "CI lower",
                "CI upper",
            ]
        ].isna().sum().sum()
    ),
)

display(
    na_sentiment_predictions.groupby(
        "Friendship network"
    ).head(2)
)

assert len(na_sentiment_predictions) == 300

assert (
    na_sentiment_predictions[
        [
            "Predicted sentiment",
            "CI lower",
            "CI upper",
        ]
    ].isna().sum().sum()
    == 0
)

print(
    "\nNegative-mood sentiment "
    "prediction data created successfully."
)

In [ ]:
# ============================================================
# SUPPLEMENTARY FIGURE
# NA WITHIN × FRIENDS → SENTIMENT
# ============================================================

line_colors = {
    "Low: 0 friends": "#C44E52",
    f"Average: {friend_mean:.1f} friends": "#4C72B0",
    f"Higher: {friend_mean + friend_sd:.1f} friends": "#55A868",
}

x_column = "Negative mood within-person (z)"
y_column = "Predicted sentiment"

fig, ax = plt.subplots(
    figsize=(8, 6)
)

for friend_label in friend_levels_raw.keys():

    plot_data = (
        na_sentiment_predictions[
            na_sentiment_predictions[
                "Friendship network"
            ] == friend_label
        ]
        .sort_values(x_column)
        .copy()
    )

    x_values = plot_data[
        x_column
    ].to_numpy(dtype=float)

    y_values = plot_data[
        y_column
    ].to_numpy(dtype=float)

    lower_values = plot_data[
        "CI lower"
    ].to_numpy(dtype=float)

    upper_values = plot_data[
        "CI upper"
    ].to_numpy(dtype=float)

    ax.plot(
        x_values,
        y_values,
        label=friend_label,
        color=line_colors[friend_label],
        linewidth=2.2,
    )

    ax.fill_between(
        x_values,
        lower_values,
        upper_values,
        color=line_colors[friend_label],
        alpha=0.15,
    )

ax.axvline(
    x=0,
    color="grey",
    linestyle="--",
    linewidth=1,
    alpha=0.7,
)

ax.set_xlabel(
    "Within-person negative mood (z)"
)

ax.set_ylabel(
    "Predicted sentiment score"
)

ax.set_title(
    "Negative Mood and Predicted Sentiment\n"
    "by Friendship-Network Size"
)

ax.legend(
    title="Friendship network",
    frameon=False,
)

ax.grid(
    axis="y",
    alpha=0.20,
)

fig.tight_layout()

na_sentiment_figure_path = (
    FIGURES
    / "Figure_NA_within_friends_sentiment_common429.png"
)

na_sentiment_data_path = (
    FIGURES
    / "Figure_NA_within_friends_sentiment_prediction_data.csv"
)

fig.savefig(
    na_sentiment_figure_path,
    dpi=300,
    bbox_inches="tight",
)

na_sentiment_predictions.to_csv(
    na_sentiment_data_path,
    index=False,
    encoding="utf-8-sig",
)

plt.show()

print("Figure saved:", na_sentiment_figure_path)
print(
    "Figure exists:",
    na_sentiment_figure_path.exists(),
)

print("Prediction data saved:", na_sentiment_data_path)
print(
    "Prediction data exists:",
    na_sentiment_data_path.exists(),
)

print(
    "Prediction rows saved:",
    len(na_sentiment_predictions),
)

In [ ]:
# ============================================================
# SUPPLEMENTARY FIGURE — PA WITHIN × FRIENDS → SENTIMENT
# CREATE PREDICTION DATA
# ============================================================

pa_sentiment_model = family2_models["PA_sent_score"]

mood_grid = np.linspace(
    family2_data["mood_PA_within_z"].min(),
    family2_data["mood_PA_within_z"].max(),
    100,
)

fixed_effect_names = (
    pa_sentiment_model.fe_params.index.tolist()
)

fixed_covariance = (
    pa_sentiment_model.cov_params()
    .loc[fixed_effect_names, fixed_effect_names]
)

beta = (
    pa_sentiment_model.fe_params
    .loc[fixed_effect_names]
    .to_numpy()
)

prediction_rows = []

for friend_label, raw_friend_value in friend_levels_raw.items():

    friend_z = (
        raw_friend_value - friend_mean
    ) / friend_sd

    for mood_value in mood_grid:

        predictor_values = {
            "Intercept": 1.0,
            "mood_PA_within_z": mood_value,
            "mood_PA_between_z": 0.0,
            "n_friends_z_person": friend_z,
            (
                "mood_PA_within_z:"
                "n_friends_z_person"
            ): mood_value * friend_z,
            (
                "mood_PA_between_z:"
                "n_friends_z_person"
            ): 0.0,
            "gender_female": 0.0,
            "family_ses_rev_z": 0.0,
            "nwords_z": 0.0,
        }

        design_vector = np.array([
            predictor_values[name]
            for name in fixed_effect_names
        ])

        predicted_value = float(
            design_vector @ beta
        )

        prediction_variance = float(
            design_vector
            @ fixed_covariance.to_numpy()
            @ design_vector
        )

        prediction_se = np.sqrt(
            max(prediction_variance, 0.0)
        )

        prediction_rows.append({
            "Positive mood within-person (z)": mood_value,
            "Predicted sentiment": predicted_value,
            "CI lower": (
                predicted_value
                - 1.96 * prediction_se
            ),
            "CI upper": (
                predicted_value
                + 1.96 * prediction_se
            ),
            "Friendship network": friend_label,
            "Raw friend count": raw_friend_value,
            "Friend count z": friend_z,
        })

pa_sentiment_predictions = pd.DataFrame(
    prediction_rows
)

print(
    "Prediction rows:",
    len(pa_sentiment_predictions),
)

print(
    "Friendship levels:",
    pa_sentiment_predictions[
        "Friendship network"
    ].unique().tolist(),
)

print(
    "Missing prediction values:",
    int(
        pa_sentiment_predictions[
            [
                "Predicted sentiment",
                "CI lower",
                "CI upper",
            ]
        ].isna().sum().sum()
    ),
)

display(
    pa_sentiment_predictions.groupby(
        "Friendship network"
    ).head(2)
)

assert len(pa_sentiment_predictions) == 300

assert (
    pa_sentiment_predictions[
        [
            "Predicted sentiment",
            "CI lower",
            "CI upper",
        ]
    ].isna().sum().sum()
    == 0
)

print(
    "\nPositive-mood sentiment "
    "prediction data created successfully."
)

In [ ]:
# ============================================================
# SUPPLEMENTARY FIGURE
# PA WITHIN × FRIENDS → SENTIMENT
# ============================================================

x_column = "Positive mood within-person (z)"
y_column = "Predicted sentiment"

fig, ax = plt.subplots(
    figsize=(8, 6)
)

for friend_label in friend_levels_raw.keys():

    plot_data = (
        pa_sentiment_predictions[
            pa_sentiment_predictions[
                "Friendship network"
            ] == friend_label
        ]
        .sort_values(x_column)
        .copy()
    )

    x_values = plot_data[
        x_column
    ].to_numpy(dtype=float)

    y_values = plot_data[
        y_column
    ].to_numpy(dtype=float)

    lower_values = plot_data[
        "CI lower"
    ].to_numpy(dtype=float)

    upper_values = plot_data[
        "CI upper"
    ].to_numpy(dtype=float)

    ax.plot(
        x_values,
        y_values,
        label=friend_label,
        color=line_colors[friend_label],
        linewidth=2.2,
    )

    ax.fill_between(
        x_values,
        lower_values,
        upper_values,
        color=line_colors[friend_label],
        alpha=0.15,
    )

ax.axvline(
    x=0,
    color="grey",
    linestyle="--",
    linewidth=1,
    alpha=0.7,
)

ax.set_xlabel(
    "Within-person positive mood (z)"
)

ax.set_ylabel(
    "Predicted sentiment score"
)

ax.set_title(
    "Positive Mood and Predicted Sentiment\n"
    "by Friendship-Network Size"
)

ax.legend(
    title="Friendship network",
    frameon=False,
)

ax.grid(
    axis="y",
    alpha=0.20,
)

fig.tight_layout()

pa_sentiment_figure_path = (
    FIGURES
    / "Figure_PA_within_friends_sentiment_common429.png"
)

pa_sentiment_data_path = (
    FIGURES
    / "Figure_PA_within_friends_sentiment_prediction_data.csv"
)

fig.savefig(
    pa_sentiment_figure_path,
    dpi=300,
    bbox_inches="tight",
)

pa_sentiment_predictions.to_csv(
    pa_sentiment_data_path,
    index=False,
    encoding="utf-8-sig",
)

plt.show()

print("Figure saved:", pa_sentiment_figure_path)
print(
    "Figure exists:",
    pa_sentiment_figure_path.exists(),
)

print("Prediction data saved:", pa_sentiment_data_path)
print(
    "Prediction data exists:",
    pa_sentiment_data_path.exists(),
)

print(
    "Prediction rows saved:",
    len(pa_sentiment_predictions),
)

In [ ]:
supplementary_sentiment_files = {
    "Negative-mood sentiment figure":
        FIGURES / "Figure_NA_within_friends_sentiment_common429.png",

    "Negative-mood prediction data":
        FIGURES / "Figure_NA_within_friends_sentiment_prediction_data.csv",

    "Positive-mood sentiment figure":
        FIGURES / "Figure_PA_within_friends_sentiment_common429.png",

    "Positive-mood prediction data":
        FIGURES / "Figure_PA_within_friends_sentiment_prediction_data.csv",
}

file_check_rows = []

for output_name, output_path in supplementary_sentiment_files.items():

    file_check_rows.append({
        "output": output_name,
        "exists": output_path.exists(),
        "size_bytes": (
            output_path.stat().st_size
            if output_path.exists()
            else 0
        ),
        "path": str(output_path),
    })

supplementary_sentiment_file_check = pd.DataFrame(
    file_check_rows
)

display(supplementary_sentiment_file_check)

assert supplementary_sentiment_file_check[
    "exists"
].all()

assert (
    supplementary_sentiment_file_check[
        "size_bytes"
    ] > 0
).all()

print(
    "\nFinal check passed: both sentiment figures "
    "and both prediction-data files exist and are non-empty."
)

In [ ]:
# ============================================================
# SUPPLEMENTARY FIGURE
# NA WITHIN × FRIENDS → NEGATIVE-EMOTION DENSITY
# CREATE PREDICTION DATA
# ============================================================

na_neg_density_model = family2_models[
    "NA_neg_emotion_density"
]

na_neg_density_mood_grid = np.linspace(
    family2_data["mood_NA_within_z"].min(),
    family2_data["mood_NA_within_z"].max(),
    100,
)

fixed_effect_names = (
    na_neg_density_model.fe_params.index.tolist()
)

fixed_effect_beta = (
    na_neg_density_model.fe_params
    .loc[fixed_effect_names]
    .to_numpy()
)

fixed_effect_covariance = (
    na_neg_density_model.cov_params()
    .loc[fixed_effect_names, fixed_effect_names]
    .to_numpy()
)

prediction_rows = []

for friend_label, raw_friend_value in friend_levels_raw.items():

    friend_z = (
        raw_friend_value - friend_mean
    ) / friend_sd

    for mood_value in na_neg_density_mood_grid:

        predictor_values = {
            "Intercept": 1.0,
            "mood_NA_within_z": mood_value,
            "mood_NA_between_z": 0.0,
            "n_friends_z_person": friend_z,
            (
                "mood_NA_within_z:"
                "n_friends_z_person"
            ): mood_value * friend_z,
            (
                "mood_NA_between_z:"
                "n_friends_z_person"
            ): 0.0,
            "gender_female": 0.0,
            "family_ses_rev_z": 0.0,
            "nwords_z": 0.0,
        }

        design_vector = np.array([
            predictor_values[name]
            for name in fixed_effect_names
        ])

        predicted_value = float(
            design_vector @ fixed_effect_beta
        )

        prediction_variance = float(
            design_vector
            @ fixed_effect_covariance
            @ design_vector
        )

        prediction_se = np.sqrt(
            max(prediction_variance, 0.0)
        )

        prediction_rows.append({
            "Negative mood within-person (z)": mood_value,
            "Predicted negative-emotion density": predicted_value,
            "CI lower": (
                predicted_value
                - 1.96 * prediction_se
            ),
            "CI upper": (
                predicted_value
                + 1.96 * prediction_se
            ),
            "Friendship network": friend_label,
            "Raw friend count": raw_friend_value,
            "Friend count z": friend_z,
        })

na_neg_density_predictions = pd.DataFrame(
    prediction_rows
)

print(
    "Model converged:",
    bool(na_neg_density_model.converged),
)

print(
    "Prediction rows:",
    len(na_neg_density_predictions),
)

print(
    "Friendship levels:",
    na_neg_density_predictions[
        "Friendship network"
    ].unique().tolist(),
)

print(
    "Missing prediction values:",
    int(
        na_neg_density_predictions[
            [
                "Predicted negative-emotion density",
                "CI lower",
                "CI upper",
            ]
        ].isna().sum().sum()
    ),
)

print(
    "Predicted-value range:",
    (
        na_neg_density_predictions[
            "Predicted negative-emotion density"
        ].min(),
        na_neg_density_predictions[
            "Predicted negative-emotion density"
        ].max(),
    ),
)

assert na_neg_density_model.converged
assert len(na_neg_density_predictions) == 300

assert (
    na_neg_density_predictions[
        [
            "Predicted negative-emotion density",
            "CI lower",
            "CI upper",
        ]
    ].isna().sum().sum()
    == 0
)

print(
    "\nNegative-mood negative-emotion-density "
    "prediction data created successfully."
)

In [ ]:
# ============================================================
# SUPPLEMENTARY FIGURE
# NA WITHIN × FRIENDS → NEGATIVE-EMOTION DENSITY
# ============================================================

x_column = "Negative mood within-person (z)"
y_column = "Predicted negative-emotion density"

fig, ax = plt.subplots(figsize=(8, 6))

for friend_label in friend_levels_raw.keys():

    plot_data = (
        na_neg_density_predictions[
            na_neg_density_predictions[
                "Friendship network"
            ] == friend_label
        ]
        .sort_values(x_column)
        .copy()
    )

    x_values = plot_data[
        x_column
    ].to_numpy(dtype=float)

    y_values = plot_data[
        y_column
    ].to_numpy(dtype=float)

    lower_values = plot_data[
        "CI lower"
    ].to_numpy(dtype=float)

    upper_values = plot_data[
        "CI upper"
    ].to_numpy(dtype=float)

    ax.plot(
        x_values,
        y_values,
        label=friend_label,
        color=line_colors[friend_label],
        linewidth=2.2,
    )

    ax.fill_between(
        x_values,
        lower_values,
        upper_values,
        color=line_colors[friend_label],
        alpha=0.15,
    )

ax.axvline(
    x=0,
    color="grey",
    linestyle="--",
    linewidth=1,
    alpha=0.7,
)

ax.axhline(
    y=0,
    color="grey",
    linestyle=":",
    linewidth=1,
    alpha=0.6,
)

ax.set_xlabel(
    "Within-person negative mood (z)"
)

ax.set_ylabel(
    "Predicted negative-emotion density"
)

ax.set_title(
    "Negative Mood and Predicted Negative-Emotion Density\n"
    "by Friendship-Network Size"
)

ax.legend(
    title="Friendship network",
    frameon=False,
)

ax.grid(
    axis="y",
    alpha=0.20,
)

fig.tight_layout()

na_neg_density_figure_path = (
    FIGURES
    / "Figure_NA_within_friends_neg_emotion_density_common429.png"
)

na_neg_density_data_path = (
    FIGURES
    / "Figure_NA_within_friends_neg_emotion_density_prediction_data.csv"
)

fig.savefig(
    na_neg_density_figure_path,
    dpi=300,
    bbox_inches="tight",
)

na_neg_density_predictions.to_csv(
    na_neg_density_data_path,
    index=False,
    encoding="utf-8-sig",
)

plt.show()

print("Figure saved:", na_neg_density_figure_path)
print(
    "Figure exists:",
    na_neg_density_figure_path.exists(),
)

print("Prediction data saved:", na_neg_density_data_path)
print(
    "Prediction data exists:",
    na_neg_density_data_path.exists(),
)

print(
    "Prediction rows saved:",
    len(na_neg_density_predictions),
)

In [ ]:
# ============================================================
# SUPPLEMENTARY FIGURE
# PA WITHIN × FRIENDS → NEGATIVE-EMOTION DENSITY
# CREATE PREDICTION DATA
# ============================================================

pa_neg_density_model = family2_models[
    "PA_neg_emotion_density"
]

pa_neg_density_mood_grid = np.linspace(
    family2_data["mood_PA_within_z"].min(),
    family2_data["mood_PA_within_z"].max(),
    100,
)

fixed_effect_names = (
    pa_neg_density_model.fe_params.index.tolist()
)

fixed_effect_beta = (
    pa_neg_density_model.fe_params
    .loc[fixed_effect_names]
    .to_numpy()
)

fixed_effect_covariance = (
    pa_neg_density_model.cov_params()
    .loc[fixed_effect_names, fixed_effect_names]
    .to_numpy()
)

prediction_rows = []

for friend_label, raw_friend_value in friend_levels_raw.items():

    friend_z = (
        raw_friend_value - friend_mean
    ) / friend_sd

    for mood_value in pa_neg_density_mood_grid:

        predictor_values = {
            "Intercept": 1.0,
            "mood_PA_within_z": mood_value,
            "mood_PA_between_z": 0.0,
            "n_friends_z_person": friend_z,
            (
                "mood_PA_within_z:"
                "n_friends_z_person"
            ): mood_value * friend_z,
            (
                "mood_PA_between_z:"
                "n_friends_z_person"
            ): 0.0,
            "gender_female": 0.0,
            "family_ses_rev_z": 0.0,
            "nwords_z": 0.0,
        }

        design_vector = np.array([
            predictor_values[name]
            for name in fixed_effect_names
        ])

        predicted_value = float(
            design_vector @ fixed_effect_beta
        )

        prediction_variance = float(
            design_vector
            @ fixed_effect_covariance
            @ design_vector
        )

        prediction_se = np.sqrt(
            max(prediction_variance, 0.0)
        )

        prediction_rows.append({
            "Positive mood within-person (z)": mood_value,
            "Predicted negative-emotion density": predicted_value,
            "CI lower": (
                predicted_value
                - 1.96 * prediction_se
            ),
            "CI upper": (
                predicted_value
                + 1.96 * prediction_se
            ),
            "Friendship network": friend_label,
            "Raw friend count": raw_friend_value,
            "Friend count z": friend_z,
        })

pa_neg_density_predictions = pd.DataFrame(
    prediction_rows
)

print(
    "Model converged:",
    bool(pa_neg_density_model.converged),
)

print(
    "Prediction rows:",
    len(pa_neg_density_predictions),
)

print(
    "Friendship levels:",
    pa_neg_density_predictions[
        "Friendship network"
    ].unique().tolist(),
)

print(
    "Missing prediction values:",
    int(
        pa_neg_density_predictions[
            [
                "Predicted negative-emotion density",
                "CI lower",
                "CI upper",
            ]
        ].isna().sum().sum()
    ),
)

print(
    "Predicted-value range:",
    (
        pa_neg_density_predictions[
            "Predicted negative-emotion density"
        ].min(),
        pa_neg_density_predictions[
            "Predicted negative-emotion density"
        ].max(),
    ),
)

assert pa_neg_density_model.converged
assert len(pa_neg_density_predictions) == 300

assert (
    pa_neg_density_predictions[
        [
            "Predicted negative-emotion density",
            "CI lower",
            "CI upper",
        ]
    ].isna().sum().sum()
    == 0
)

print(
    "\nPositive-mood negative-emotion-density "
    "prediction data created successfully."
)

In [ ]:
# ============================================================
# SUPPLEMENTARY FIGURE
# PA WITHIN × FRIENDS → NEGATIVE-EMOTION DENSITY
# ============================================================

x_column = "Positive mood within-person (z)"
y_column = "Predicted negative-emotion density"

fig, ax = plt.subplots(figsize=(8, 6))

for friend_label in friend_levels_raw.keys():

    plot_data = (
        pa_neg_density_predictions[
            pa_neg_density_predictions[
                "Friendship network"
            ] == friend_label
        ]
        .sort_values(x_column)
        .copy()
    )

    x_values = plot_data[
        x_column
    ].to_numpy(dtype=float)

    y_values = plot_data[
        y_column
    ].to_numpy(dtype=float)

    lower_values = plot_data[
        "CI lower"
    ].to_numpy(dtype=float)

    upper_values = plot_data[
        "CI upper"
    ].to_numpy(dtype=float)

    ax.plot(
        x_values,
        y_values,
        label=friend_label,
        color=line_colors[friend_label],
        linewidth=2.2,
    )

    ax.fill_between(
        x_values,
        lower_values,
        upper_values,
        color=line_colors[friend_label],
        alpha=0.15,
    )

ax.axvline(
    x=0,
    color="grey",
    linestyle="--",
    linewidth=1,
    alpha=0.7,
)

ax.axhline(
    y=0,
    color="grey",
    linestyle=":",
    linewidth=1,
    alpha=0.6,
)

ax.set_xlabel(
    "Within-person positive mood (z)"
)

ax.set_ylabel(
    "Predicted negative-emotion density"
)

ax.set_title(
    "Positive Mood and Predicted Negative-Emotion Density\n"
    "by Friendship-Network Size"
)

ax.legend(
    title="Friendship network",
    frameon=False,
)

ax.grid(
    axis="y",
    alpha=0.20,
)

fig.tight_layout()

pa_neg_density_figure_path = (
    FIGURES
    / "Figure_PA_within_friends_neg_emotion_density_common429.png"
)

pa_neg_density_data_path = (
    FIGURES
    / "Figure_PA_within_friends_neg_emotion_density_prediction_data.csv"
)

fig.savefig(
    pa_neg_density_figure_path,
    dpi=300,
    bbox_inches="tight",
)

pa_neg_density_predictions.to_csv(
    pa_neg_density_data_path,
    index=False,
    encoding="utf-8-sig",
)

plt.show()

print("Figure saved:", pa_neg_density_figure_path)
print(
    "Figure exists:",
    pa_neg_density_figure_path.exists(),
)

print("Prediction data saved:", pa_neg_density_data_path)
print(
    "Prediction data exists:",
    pa_neg_density_data_path.exists(),
)

print(
    "Prediction rows saved:",
    len(pa_neg_density_predictions),
)

In [ ]:
# ============================================================
# FINAL MODELS 8 — INSPECT FAMILY 2 WARNINGS
# ============================================================

warning_check = family2_health[
    family2_health["boundary_or_singular_warning"] == True
][
    [
        "model_id",
        "optimizer",
        "converged",
        "participant_intercept_variance",
        "residual_variance",
        "ICC",
        "warning_text",
    ]
].copy()

pd.set_option("display.max_colwidth", None)

display(warning_check)

pd.reset_option("display.max_colwidth")

In [ ]:
# ============================================================
# FINAL MODELS 9 — INSPECT FAMILY 2 INTERACTION ESTIMATES
# ============================================================

family2_result_rows = []

for mood_family, mood_terms in family2_mood_families.items():

    within_term = mood_terms["within"]
    between_term = mood_terms["between"]

    interaction_within = (
        f"{within_term}:n_friends_z_person"
    )
    interaction_between = (
        f"{between_term}:n_friends_z_person"
    )

    terms_to_extract = [
        within_term,
        between_term,
        "n_friends_z_person",
        interaction_within,
        interaction_between,
        "gender_female",
        "family_ses_rev_z",
        "nwords_z",
    ]

    for outcome in family2_outcomes:

        model_id = f"{mood_family}_{outcome}"
        model = family2_models[model_id]

        participant_variance = float(
            model.cov_re.iloc[0, 0]
        )
        residual_variance = float(model.scale)

        icc = (
            participant_variance
            / (participant_variance + residual_variance)
        )

        for term in terms_to_extract:

            family2_result_rows.append({
                "model_id": model_id,
                "mood_family": mood_family,
                "outcome": outcome,
                "term": term,
                "beta": model.params[term],
                "SE": model.bse[term],
                "p": model.pvalues[term],
                "N": int(model.nobs),
                "participants": int(model.model.n_groups),
                "ICC": icc,
            })

family2_results = pd.DataFrame(
    family2_result_rows
)

# Display only the main network and interaction terms first
family2_focal = family2_results[
    family2_results["term"].str.contains(
        "n_friends_z_person",
        regex=False,
    )
].copy()

family2_focal[
    ["beta", "SE", "p", "ICC"]
] = family2_focal[
    ["beta", "SE", "p", "ICC"]
].round(4)

print("Family 2 focal network and interaction estimates:")
display(family2_focal)

In [ ]:
# ============================================================
# FINAL MODELS 10 — FAMILY 2 CLUSTER-ROBUST CHECK
# ============================================================

family2_cluster_models = {}
family2_robustness_rows = []

for mood_family, mood_terms in family2_mood_families.items():

    within_term = mood_terms["within"]
    between_term = mood_terms["between"]

    focal_terms = [
        "n_friends_z_person",
        f"{within_term}:n_friends_z_person",
        f"{between_term}:n_friends_z_person",
    ]

    for outcome in family2_outcomes:

        model_id = f"{mood_family}_{outcome}"

        formula = (
            f"{outcome} ~ "
            f"{within_term} * n_friends_z_person + "
            f"{between_term} * n_friends_z_person + "
            "gender_female + family_ses_rev_z + nwords_z"
        )

        cluster_model = smf.ols(
            formula=formula,
            data=family2_data,
        ).fit(
            cov_type="cluster",
            cov_kwds={
                "groups": family2_data["participantid"]
            },
        )

        family2_cluster_models[model_id] = cluster_model
        mixed_model = family2_models[model_id]

        for term in focal_terms:

            family2_robustness_rows.append({
                "model_id": model_id,
                "mood_family": mood_family,
                "outcome": outcome,
                "term": term,
                "mixed_beta": mixed_model.params[term],
                "mixed_SE": mixed_model.bse[term],
                "mixed_p": mixed_model.pvalues[term],
                "cluster_beta": cluster_model.params[term],
                "cluster_SE": cluster_model.bse[term],
                "cluster_p": cluster_model.pvalues[term],
            })

family2_robustness = pd.DataFrame(
    family2_robustness_rows
)

family2_robustness[
    [
        "mixed_beta",
        "mixed_SE",
        "mixed_p",
        "cluster_beta",
        "cluster_SE",
        "cluster_p",
    ]
] = family2_robustness[
    [
        "mixed_beta",
        "mixed_SE",
        "mixed_p",
        "cluster_beta",
        "cluster_SE",
        "cluster_p",
    ]
].round(4)

display(family2_robustness)

In [ ]:
# ============================================================
# FINAL MODELS 11 — SAVE FAMILY 2 ROBUSTNESS RESULTS
# ============================================================

family2_robustness_path = (
    TABLES / "Family2_mixed_vs_clustered_robustness.csv"
)

family2_robustness.to_csv(
    family2_robustness_path,
    index=False,
)

print("Saved:", family2_robustness_path)
print("Rows saved:", len(family2_robustness))
print("Models compared:", family2_robustness["model_id"].nunique())
print("Common sample: 429 diary entries from 40 participants")

In [ ]:
# ============================================================
# FINAL MODELS 12 — SAVE FAMILY 2 MIXED-MODEL RESULTS
# ============================================================

family2_model_rows = []

for model_id, model in family2_models.items():

    mood_family, outcome = model_id.split("_", 1)

    participant_variance = float(
        model.cov_re.iloc[0, 0]
    )
    residual_variance = float(
        model.scale
    )

    icc = (
        participant_variance
        / (participant_variance + residual_variance)
    )

    for term in model.params.index:

        if term == "Group Var":
            continue

        family2_model_rows.append({
            "model_id": model_id,
            "mood_family": mood_family,
            "outcome": outcome,
            "term": term,
            "beta": float(model.params[term]),
            "SE": float(model.bse[term]),
            "p": float(model.pvalues[term]),
            "N_observations": int(model.nobs),
            "N_participants": int(model.model.n_groups),
            "participant_intercept_variance":
                participant_variance,
            "residual_variance":
                residual_variance,
            "ICC": icc,
        })

family2_saved_results = pd.DataFrame(
    family2_model_rows
)

family2_results_path = (
    TABLES / "Family2_mixed_models_common429.csv"
)

family2_saved_results.to_csv(
    family2_results_path,
    index=False,
)

print("Saved:", family2_results_path)
print("Rows saved:", len(family2_saved_results))
print(
    "Models saved:",
    family2_saved_results["model_id"].nunique()
)
print(
    "N observations by model:",
    family2_saved_results.groupby("model_id")[
        "N_observations"
    ].first().to_dict()
)

In [ ]:
# ============================================================
# FINAL MODELS 13 — FORMAT TABLE 4A: NEGATIVE MOOD × FRIENDS
# ============================================================

family2_outcome_labels = {
    "social_conn_llm": "LLM social connectedness",
    "sent_score": "Sentiment score",
    "social_density": "Social-word density",
    "neg_emotion_density": "Negative-emotion density",
}

family2_na_term_labels = {
    "mood_NA_within_z": "Negative mood, within-person",
    "mood_NA_between_z": "Negative mood, between-person",
    "n_friends_z_person": "Number of nominated friends",
    "mood_NA_within_z:n_friends_z_person":
        "Negative mood within × friends",
    "mood_NA_between_z:n_friends_z_person":
        "Negative mood between × friends",
    "gender_female": "Female indicator",
    "family_ses_rev_z": "Socioeconomic position",
    "nwords_z": "Diary word count",
}

def significance_symbol(p):
    if p < .001:
        return "***"
    if p < .01:
        return "**"
    if p < .05:
        return "*"
    if p < .10:
        return "†"
    return ""

def format_family2_cell(model, term):
    beta = model.params[term]
    se = model.bse[term]
    p = model.pvalues[term]

    return (
        f"{beta:.3f} "
        f"({se:.3f})"
        f"{significance_symbol(p)}"
    )

table4a_rows = []

for term, label in family2_na_term_labels.items():

    row = {"Predictor": label}

    for outcome in family2_outcomes:

        model = family2_models[
            f"NA_{outcome}"
        ]

        row[family2_outcome_labels[outcome]] = (
            format_family2_cell(model, term)
        )

    table4a_rows.append(row)

# Add model information
table4a_model_information = [
    (
        "N observations",
        lambda model: str(int(model.nobs)),
    ),
    (
        "N participants",
        lambda model: str(int(model.model.n_groups)),
    ),
    (
        "Participant intercept variance",
        lambda model: f"{float(model.cov_re.iloc[0, 0]):.6f}",
    ),
    (
        "Residual variance",
        lambda model: f"{float(model.scale):.6f}",
    ),
    (
        "ICC",
        lambda model: f"{float(model.cov_re.iloc[0, 0]) / (float(model.cov_re.iloc[0, 0]) + float(model.scale)):.3f}",
    ),
]

for label, value_function in table4a_model_information:

    row = {"Predictor": label}

    for outcome in family2_outcomes:

        model = family2_models[
            f"NA_{outcome}"
        ]

        row[family2_outcome_labels[outcome]] = (
            value_function(model)
        )

    table4a_rows.append(row)

table4a_final = pd.DataFrame(
    table4a_rows
)

table4a_path = (
    TABLES / "Table_4a_Family2_NA_friends_common429.csv"
)

table4a_final.to_csv(
    table4a_path,
    index=False,
)

print("Saved:", table4a_path)
display(table4a_final)

In [ ]:
# ============================================================
# FINAL MODELS 14 — FORMAT TABLE 4B: POSITIVE MOOD × FRIENDS
# ============================================================

family2_pa_term_labels = {
    "mood_PA_within_z": "Positive mood, within-person",
    "mood_PA_between_z": "Positive mood, between-person",
    "n_friends_z_person": "Number of nominated friends",
    "mood_PA_within_z:n_friends_z_person":
        "Positive mood within × friends",
    "mood_PA_between_z:n_friends_z_person":
        "Positive mood between × friends",
    "gender_female": "Female indicator",
    "family_ses_rev_z": "Socioeconomic position",
    "nwords_z": "Diary word count",
}

table4b_rows = []

for term, label in family2_pa_term_labels.items():

    row = {"Predictor": label}

    for outcome in family2_outcomes:

        model = family2_models[
            f"PA_{outcome}"
        ]

        row[family2_outcome_labels[outcome]] = (
            format_family2_cell(model, term)
        )

    table4b_rows.append(row)

for label, value_function in table4a_model_information:

    row = {"Predictor": label}

    for outcome in family2_outcomes:

        model = family2_models[
            f"PA_{outcome}"
        ]

        row[family2_outcome_labels[outcome]] = (
            value_function(model)
        )

    table4b_rows.append(row)

table4b_final = pd.DataFrame(
    table4b_rows
)

table4b_path = (
    TABLES / "Table_4b_Family2_PA_friends_common429.csv"
)

table4b_final.to_csv(
    table4b_path,
    index=False,
)

print("Saved:", table4b_path)
display(table4b_final)

## Final Model Decision 2 — Family 2 Friendship-Network Moderation

Family 2 was re-estimated using the common complete-case sample of 429 diary entries from 40 participants. The moderator was the participant-standardized number of nominated friends (`n_friends_z_person`). Consequently, the lower-order mood coefficients represent mood–outcome associations for participants with an average-sized friendship network.

For LLM social connectedness, friendship-network size did not moderate the association with within-person or between-person negative mood. The within-person positive mood × friends interaction was positive and marginally significant in the primary mixed model (b = 0.058, SE = 0.032, p = .076). This provides suggestive rather than conclusive evidence that the association between higher-than-usual positive mood and LLM-rated social connectedness becomes stronger among participants with larger friendship networks. The interaction had the same positive direction under participant-clustered OLS.

For sentiment, both within-person moderation effects were supported across estimators. The positive negative-mood × friends interaction indicates that the negative association between within-person negative mood and sentiment becomes weaker as friendship-network size increases. The negative positive-mood × friends interaction indicates that the positive association between within-person positive mood and sentiment also becomes weaker as friendship-network size increases. Together, these results suggest that diary sentiment is less strongly coupled with momentary mood among participants with larger friendship networks.

Social-word density showed no stable moderation effects. Several interactions involving negative-emotion density were estimated, but this outcome was highly sparse and its mixed models produced boundary or singularity warnings. These lexical-density moderation results are therefore treated as secondary and interpreted cautiously.

The earlier reverse-direction interaction model in which positive mood was the outcome is superseded by the corrected Family 2 specification and is not used as a final thesis result.
``

In [ ]:
# ============================================================
# FINAL MODELS 15 — FIT FAMILY 3 JOINT PHQ-9/GAD-7 MODELS
# ============================================================

import warnings
import statsmodels.formula.api as smf

family3_data = analysis[
    analysis["common_language_sample"]
].copy()

family3_outcomes = [
    "social_conn_llm",
    "sent_score",
    "social_density",
    "neg_emotion_density",
]

family3_predictors = [
    "PHQ9_total_z",
    "GAD7_total_z",
    "gender_female",
    "family_ses_rev_z",
    "nwords_z",
]

required_family3 = (
    ["participantid"]
    + family3_outcomes
    + family3_predictors
)

print("Family 3 rows:", len(family3_data))
print(
    "Family 3 participants:",
    family3_data["participantid"].nunique()
)
print(
    "Missing values across required variables:",
    int(
        family3_data[
            required_family3
        ].isna().sum().sum()
    )
)

print(
    "Correlation between PHQ-9 and GAD-7:",
    round(
        family3_data[
            ["PHQ9_total_z", "GAD7_total_z"]
        ].corr().iloc[0, 1],
        3,
    )
)

family3_joint_models = {}
family3_joint_health_rows = []

optimizers = [
    "lbfgs",
    "bfgs",
    "cg",
    "powell",
    "nm",
]

for outcome in family3_outcomes:

    formula = (
        f"{outcome} ~ "
        "PHQ9_total_z + GAD7_total_z + "
        "gender_female + family_ses_rev_z + nwords_z"
    )

    fitted_model = None
    optimizer_used = None
    selected_warning_messages = []
    last_error = None

    for optimizer in optimizers:

        try:
            with warnings.catch_warnings(record=True) as caught:
                warnings.simplefilter("always")

                candidate = smf.mixedlm(
                    formula=formula,
                    data=family3_data,
                    groups=family3_data["participantid"],
                ).fit(
                    reml=True,
                    method=optimizer,
                    maxiter=2000,
                    disp=False,
                )

            current_warnings = [
                str(w.message)
                for w in caught
            ]

            if candidate.converged:
                fitted_model = candidate
                optimizer_used = optimizer
                selected_warning_messages = current_warnings
                break

        except Exception as error:
            last_error = str(error)

    if fitted_model is not None:

        participant_variance = float(
            fitted_model.cov_re.iloc[0, 0]
        )
        residual_variance = float(
            fitted_model.scale
        )

        icc = (
            participant_variance
            / (
                participant_variance
                + residual_variance
            )
        )

        family3_joint_models[outcome] = fitted_model

        family3_joint_health_rows.append({
            "outcome": outcome,
            "N_observations": int(fitted_model.nobs),
            "N_participants":
                int(fitted_model.model.n_groups),
            "optimizer": optimizer_used,
            "converged": fitted_model.converged,
            "participant_intercept_variance":
                participant_variance,
            "residual_variance":
                residual_variance,
            "ICC": icc,
            "boundary_or_singular_warning": any(
                "boundary" in message.lower()
                or "singular" in message.lower()
                for message in selected_warning_messages
            ),
            "warning_text": " | ".join(
                selected_warning_messages
            ),
            "last_error": "",
        })

    else:

        family3_joint_health_rows.append({
            "outcome": outcome,
            "N_observations": len(family3_data),
            "N_participants":
                family3_data["participantid"].nunique(),
            "optimizer": None,
            "converged": False,
            "participant_intercept_variance": np.nan,
            "residual_variance": np.nan,
            "ICC": np.nan,
            "boundary_or_singular_warning": None,
            "warning_text": "",
            "last_error": last_error,
        })

family3_joint_health = pd.DataFrame(
    family3_joint_health_rows
)

family3_joint_health_display = (
    family3_joint_health[
        [
            "outcome",
            "N_observations",
            "N_participants",
            "optimizer",
            "converged",
            "participant_intercept_variance",
            "residual_variance",
            "ICC",
            "boundary_or_singular_warning",
        ]
    ].copy()
)

family3_joint_health_display[
    [
        "participant_intercept_variance",
        "residual_variance",
        "ICC",
    ]
] = family3_joint_health_display[
    [
        "participant_intercept_variance",
        "residual_variance",
        "ICC",
    ]
].round(6)

print("\nFamily 3 joint-model health check:")
display(family3_joint_health_display)

In [ ]:
# ============================================================
# FINAL MODELS 16 — INSPECT FAMILY 3 JOINT ESTIMATES
# ============================================================

family3_joint_rows = []

for outcome, model in family3_joint_models.items():

    participant_variance = float(
        model.cov_re.iloc[0, 0]
    )
    residual_variance = float(model.scale)

    icc = (
        participant_variance
        / (participant_variance + residual_variance)
    )

    for term in [
        "PHQ9_total_z",
        "GAD7_total_z",
        "gender_female",
        "family_ses_rev_z",
        "nwords_z",
    ]:

        family3_joint_rows.append({
            "outcome": outcome,
            "term": term,
            "beta": model.params[term],
            "SE": model.bse[term],
            "p": model.pvalues[term],
            "N": int(model.nobs),
            "participants": int(model.model.n_groups),
            "ICC": icc,
        })

family3_joint_results = pd.DataFrame(
    family3_joint_rows
)

family3_joint_display = (
    family3_joint_results.copy()
)

family3_joint_display[
    ["beta", "SE", "p", "ICC"]
] = family3_joint_display[
    ["beta", "SE", "p", "ICC"]
].round(4)

display(family3_joint_display)

In [ ]:
# ============================================================
# FINAL MODELS 17 — FAMILY 3 SEPARATE PHQ-9 AND GAD-7 MODELS
# ============================================================

family3_single_models = {}
family3_single_rows = []

single_predictors = {
    "PHQ_only": "PHQ9_total_z",
    "GAD_only": "GAD7_total_z",
}

for specification, focal_predictor in single_predictors.items():

    for outcome in family3_outcomes:

        model_id = f"{specification}_{outcome}"

        formula = (
            f"{outcome} ~ "
            f"{focal_predictor} + "
            "gender_female + family_ses_rev_z + nwords_z"
        )

        fitted_model = None
        optimizer_used = None
        selected_warnings = []
        last_error = None

        for optimizer in optimizers:

            try:
                with warnings.catch_warnings(record=True) as caught:
                    warnings.simplefilter("always")

                    candidate = smf.mixedlm(
                        formula=formula,
                        data=family3_data,
                        groups=family3_data["participantid"],
                    ).fit(
                        reml=True,
                        method=optimizer,
                        maxiter=2000,
                        disp=False,
                    )

                current_warnings = [
                    str(w.message)
                    for w in caught
                ]

                if candidate.converged:
                    fitted_model = candidate
                    optimizer_used = optimizer
                    selected_warnings = current_warnings
                    break

            except Exception as error:
                last_error = str(error)

        if fitted_model is not None:

            family3_single_models[model_id] = fitted_model

            participant_variance = float(
                fitted_model.cov_re.iloc[0, 0]
            )
            residual_variance = float(
                fitted_model.scale
            )

            family3_single_rows.append({
                "model_id": model_id,
                "specification": specification,
                "outcome": outcome,
                "predictor": focal_predictor,
                "beta": fitted_model.params[focal_predictor],
                "SE": fitted_model.bse[focal_predictor],
                "p": fitted_model.pvalues[focal_predictor],
                "N": int(fitted_model.nobs),
                "participants":
                    int(fitted_model.model.n_groups),
                "optimizer": optimizer_used,
                "participant_intercept_variance":
                    participant_variance,
                "residual_variance":
                    residual_variance,
                "boundary_or_singular_warning": any(
                    "boundary" in message.lower()
                    or "singular" in message.lower()
                    for message in selected_warnings
                ),
            })

        else:

            family3_single_rows.append({
                "model_id": model_id,
                "specification": specification,
                "outcome": outcome,
                "predictor": focal_predictor,
                "beta": np.nan,
                "SE": np.nan,
                "p": np.nan,
                "N": len(family3_data),
                "participants":
                    family3_data["participantid"].nunique(),
                "optimizer": None,
                "participant_intercept_variance": np.nan,
                "residual_variance": np.nan,
                "boundary_or_singular_warning": None,
            })

family3_single_results = pd.DataFrame(
    family3_single_rows
)

family3_single_display = (
    family3_single_results.copy()
)

family3_single_display[
    [
        "beta",
        "SE",
        "p",
        "participant_intercept_variance",
        "residual_variance",
    ]
] = family3_single_display[
    [
        "beta",
        "SE",
        "p",
        "participant_intercept_variance",
        "residual_variance",
    ]
].round(4)

display(family3_single_display)


In [ ]:
# ============================================================
# FINAL MODELS 18 — SAVE FAMILY 3 SEPARATE-MODEL RESULTS
# ============================================================

family3_single_path = (
    TABLES / "Family3_PHQ_only_GAD_only_common429.csv"
)

family3_single_results.to_csv(
    family3_single_path,
    index=False,
)

print("Saved:", family3_single_path)
print("Rows saved:", len(family3_single_results))
print(
    "Specifications:",
    family3_single_results["specification"]
    .unique()
    .tolist()
)
print("Common sample: 429 diary entries from 40 participants")

In [ ]:
# ============================================================
# FINAL MODELS 19 — ROBUSTNESS OF SEPARATE PHQ-9/GAD-7 MODELS
# ============================================================

family3_single_robustness_rows = []

for specification, focal_predictor in single_predictors.items():

    for outcome in family3_outcomes:

        model_id = f"{specification}_{outcome}"

        formula = (
            f"{outcome} ~ "
            f"{focal_predictor} + "
            "gender_female + family_ses_rev_z + nwords_z"
        )

        cluster_model = smf.ols(
            formula=formula,
            data=family3_data,
        ).fit(
            cov_type="cluster",
            cov_kwds={
                "groups": family3_data["participantid"]
            },
        )

        mixed_model = family3_single_models[model_id]

        family3_single_robustness_rows.append({
            "model_id": model_id,
            "specification": specification,
            "outcome": outcome,
            "predictor": focal_predictor,
            "mixed_beta": mixed_model.params[focal_predictor],
            "mixed_SE": mixed_model.bse[focal_predictor],
            "mixed_p": mixed_model.pvalues[focal_predictor],
            "cluster_beta": cluster_model.params[focal_predictor],
            "cluster_SE": cluster_model.bse[focal_predictor],
            "cluster_p": cluster_model.pvalues[focal_predictor],
        })

family3_single_robustness = pd.DataFrame(
    family3_single_robustness_rows
)

family3_single_robustness[
    [
        "mixed_beta",
        "mixed_SE",
        "mixed_p",
        "cluster_beta",
        "cluster_SE",
        "cluster_p",
    ]
] = family3_single_robustness[
    [
        "mixed_beta",
        "mixed_SE",
        "mixed_p",
        "cluster_beta",
        "cluster_SE",
        "cluster_p",
    ]
].round(4)

display(family3_single_robustness)

In [ ]:
# ============================================================
# FINAL MODELS 20 — SAVE FAMILY 3 ROBUSTNESS RESULTS
# ============================================================

family3_single_robustness_path = (
    TABLES / "Family3_separate_models_mixed_vs_clustered.csv"
)

family3_single_robustness.to_csv(
    family3_single_robustness_path,
    index=False,
)

print("Saved:", family3_single_robustness_path)
print("Rows saved:", len(family3_single_robustness))
print("Common sample: 429 diary entries from 40 participants")

In [ ]:
# ============================================================
# FINAL MODELS 21 — SAVE FAMILY 3 JOINT-MODEL RESULTS
# ============================================================

family3_joint_saved_rows = []

for outcome, model in family3_joint_models.items():

    participant_variance = float(
        model.cov_re.iloc[0, 0]
    )
    residual_variance = float(model.scale)

    icc = (
        participant_variance
        / (participant_variance + residual_variance)
    )

    for term in model.params.index:

        if term == "Group Var":
            continue

        family3_joint_saved_rows.append({
            "outcome": outcome,
            "term": term,
            "beta": float(model.params[term]),
            "SE": float(model.bse[term]),
            "p": float(model.pvalues[term]),
            "N_observations": int(model.nobs),
            "N_participants": int(model.model.n_groups),
            "participant_intercept_variance":
                participant_variance,
            "residual_variance":
                residual_variance,
            "ICC": icc,
        })

family3_joint_saved = pd.DataFrame(
    family3_joint_saved_rows
)

family3_joint_path = (
    TABLES / "Family3_joint_PHQ_GAD_common429.csv"
)

family3_joint_saved.to_csv(
    family3_joint_path,
    index=False,
)

print("Saved:", family3_joint_path)
print("Rows saved:", len(family3_joint_saved))
print(
    "Outcomes:",
    family3_joint_saved["outcome"].nunique()
)
print(
    "N observations by outcome:",
    family3_joint_saved.groupby("outcome")[
        "N_observations"
    ].first().to_dict()
)

In [ ]:
# ============================================================
# FINAL MODELS 22 — SAVE COMPLETE FAMILY 3 SEPARATE MODELS
# ============================================================

family3_single_full_rows = []

for model_id, model in family3_single_models.items():

    specification, outcome = model_id.split("_", 1)

    # Correct split for PHQ_only_... and GAD_only_...
    if model_id.startswith("PHQ_only_"):
        specification = "PHQ_only"
        outcome = model_id.replace("PHQ_only_", "")
    elif model_id.startswith("GAD_only_"):
        specification = "GAD_only"
        outcome = model_id.replace("GAD_only_", "")

    participant_variance = float(
        model.cov_re.iloc[0, 0]
    )

    residual_variance = float(
        model.scale
    )

    icc = (
        participant_variance
        / (participant_variance + residual_variance)
    )

    for term in model.params.index:

        if term == "Group Var":
            continue

        family3_single_full_rows.append({
            "model_id": model_id,
            "specification": specification,
            "outcome": outcome,
            "term": term,
            "beta": float(model.params[term]),
            "SE": float(model.bse[term]),
            "p": float(model.pvalues[term]),
            "N_observations": int(model.nobs),
            "N_participants": int(model.model.n_groups),
            "participant_intercept_variance":
                participant_variance,
            "residual_variance":
                residual_variance,
            "ICC": icc,
        })

family3_single_full = pd.DataFrame(
    family3_single_full_rows
)

family3_single_full_path = (
    TABLES / "Family3_separate_models_full_common429.csv"
)

family3_single_full.to_csv(
    family3_single_full_path,
    index=False,
)

print("Saved:", family3_single_full_path)
print("Rows saved:", len(family3_single_full))
print(
    "Models saved:",
    family3_single_full["model_id"].nunique()
)
print(
    "Specifications:",
    family3_single_full["specification"]
    .unique()
    .tolist()
)

In [ ]:
# ============================================================
# FINAL MODELS 23 — FORMAT PRIMARY TABLE 5
# SEPARATE PHQ-9 AND GAD-7 MODELS
# ============================================================

family3_outcome_labels = {
    "social_conn_llm": "LLM social connectedness",
    "sent_score": "Sentiment score",
    "social_density": "Social-word density",
    "neg_emotion_density": "Negative-emotion density",
}

family3_control_labels = {
    "gender_female": "Female indicator",
    "family_ses_rev_z": "Socioeconomic position",
    "nwords_z": "Diary word count",
}

def significance_symbol(p):
    if p < .001:
        return "***"
    if p < .01:
        return "**"
    if p < .05:
        return "*"
    if p < .10:
        return "†"
    return ""

def format_family3_cell(model, term):
    beta = model.params[term]
    se = model.bse[term]
    p = model.pvalues[term]

    return (
        f"{beta:.3f} "
        f"({se:.3f})"
        f"{significance_symbol(p)}"
    )

table5_rows = []

# ------------------------------------------------------------
# PANEL A — PHQ-9 ONLY
# ------------------------------------------------------------

table5_rows.append({
    "Predictor": "Panel A — PHQ-9-only models",
    **{
        label: ""
        for label in family3_outcome_labels.values()
    }
})

phq_terms = {
    "PHQ9_total_z": "PHQ-9 depression",
    **family3_control_labels,
}

for term, label in phq_terms.items():

    row = {"Predictor": label}

    for outcome in family3_outcomes:

        model = family3_single_models[
            f"PHQ_only_{outcome}"
        ]

        row[family3_outcome_labels[outcome]] = (
            format_family3_cell(model, term)
        )

    table5_rows.append(row)

# Model information for Panel A
for information_label, value_function in [
    (
        "N observations",
        lambda model: str(int(model.nobs)),
    ),
    (
        "N participants",
        lambda model: str(int(model.model.n_groups)),
    ),
    (
        "Participant intercept variance",
        lambda model: f"{float(model.cov_re.iloc[0, 0]):.6f}",
    ),
    (
        "Residual variance",
        lambda model: f"{float(model.scale):.6f}",
    ),
    (
        "ICC",
        lambda model: f"{float(model.cov_re.iloc[0, 0]) / (float(model.cov_re.iloc[0, 0]) + float(model.scale)):.3f}",
    ),
]:

    row = {"Predictor": information_label}

    for outcome in family3_outcomes:

        model = family3_single_models[
            f"PHQ_only_{outcome}"
        ]

        row[family3_outcome_labels[outcome]] = (
            value_function(model)
        )

    table5_rows.append(row)

# ------------------------------------------------------------
# PANEL B — GAD-7 ONLY
# ------------------------------------------------------------

table5_rows.append({
    "Predictor": "Panel B — GAD-7-only models",
    **{
        label: ""
        for label in family3_outcome_labels.values()
    }
})

gad_terms = {
    "GAD7_total_z": "GAD-7 anxiety",
    **family3_control_labels,
}

for term, label in gad_terms.items():

    row = {"Predictor": label}

    for outcome in family3_outcomes:

        model = family3_single_models[
            f"GAD_only_{outcome}"
        ]

        row[family3_outcome_labels[outcome]] = (
            format_family3_cell(model, term)
        )

    table5_rows.append(row)

# Model information for Panel B
for information_label, value_function in [
    (
        "N observations",
        lambda model: str(int(model.nobs)),
    ),
    (
        "N participants",
        lambda model: str(int(model.model.n_groups)),
    ),
    (
        "Participant intercept variance",
        lambda model: f"{float(model.cov_re.iloc[0, 0]):.6f}",
    ),
    (
        "Residual variance",
        lambda model: f"{float(model.scale):.6f}",
    ),
    (
        "ICC",
        lambda model: f"{float(model.cov_re.iloc[0, 0]) / (float(model.cov_re.iloc[0, 0]) + float(model.scale)):.3f}",
    ),
]:

    row = {"Predictor": information_label}

    for outcome in family3_outcomes:

        model = family3_single_models[
            f"GAD_only_{outcome}"
        ]

        row[family3_outcome_labels[outcome]] = (
            value_function(model)
        )

    table5_rows.append(row)

table5_final = pd.DataFrame(table5_rows)

table5_path = (
    TABLES / "Table_5_Family3_separate_PHQ_GAD_common429.csv"
)

table5_final.to_csv(
    table5_path,
    index=False,
)

print("Saved:", table5_path)
display(table5_final)

## Final Model Decision 3 — Family 3 Depression and Anxiety Models

Family 3 was estimated using the common complete-case sample of 429 diary entries from 40 participants. Because PHQ-9 depression and GAD-7 anxiety were strongly correlated (r = .768), models containing both predictors jointly were supplemented by separate PHQ-9-only and GAD-7-only specifications.

In the joint models, neither PHQ-9 nor GAD-7 showed a clear unique association with any of the four language-based outcomes. When estimated separately, neither predictor was associated with LLM social connectedness or social-word density. However, both PHQ-9 and GAD-7 showed clear negative associations with sentiment. These sentiment associations were also supported using participant-clustered robust standard errors.

The separate-model findings indicate that greater depression and anxiety symptoms are each associated with less-positive diary sentiment. Because PHQ-9 and GAD-7 share substantial variance, these results should not be interpreted as distinct or independent depression-specific and anxiety-specific effects. Rather, their shared symptom variance appears to account for the sentiment association that could not be uniquely assigned to either predictor in the joint model.

Positive associations between PHQ-9 or GAD-7 and negative-emotion density were marginally significant in the mixed models but were not supported at the .10 level using participant-clustered robust standard errors. Given the sparse distribution and boundary warnings for negative-emotion density, these patterns are treated as secondary and uncertain.

The separate PHQ-9-only and GAD-7-only specifications are used as the primary Family 3 results, following the supervisor’s recommendation. The joint PHQ-9/GAD-7 models are retained as a supplementary robustness specification.

In [ ]:
# ============================================================
# FINAL FIGURE 1 — POSITIVE MOOD × FRIENDS
# Outcome: LLM social connectedness
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Exact final audited model
pa_llm_model = family2_models["PA_social_conn_llm"]

# Participant-level friendship reference values
friend_mean = friends_mean_person       # 2.80
friend_sd = friends_sd_person           # 2.82

# Raw friend-count levels for the plot
friend_levels_raw = {
    "Low: 0 friends": 0,
    f"Average: {friend_mean:.1f} friends": friend_mean,
    f"Higher: {friend_mean + friend_sd:.1f} friends": (
        friend_mean + friend_sd
    ),
}

# Convert raw friend counts to the participant-standardized scale
friend_levels_z = {
    label: (raw_value - friend_mean) / friend_sd
    for label, raw_value in friend_levels_raw.items()
}

# Use the central 90% of observed within-person positive mood
# to avoid extending the figure into sparsely observed extremes
pa_grid = np.linspace(
    family2_data["mood_PA_within_z"].quantile(0.05),
    family2_data["mood_PA_within_z"].quantile(0.95),
    100,
)

# Hold controls at their reference/mean values
pa_between_reference = 0
ses_reference = 0
word_count_reference = 0
gender_reference = family2_data["gender_female"].mean()

# Fixed-effect coefficient names
fixed_effect_names = [
    name
    for name in pa_llm_model.fe_params.index
]

beta = pa_llm_model.fe_params.loc[
    fixed_effect_names
].to_numpy()

fixed_covariance = pa_llm_model.cov_params().loc[
    fixed_effect_names,
    fixed_effect_names,
].to_numpy()

prediction_rows = []

for friend_label, friend_z in friend_levels_z.items():

    for mood_value in pa_grid:

        values = {
            "Intercept": 1.0,
            "mood_PA_within_z": mood_value,
            "n_friends_z_person": friend_z,
            "mood_PA_within_z:n_friends_z_person":
                mood_value * friend_z,
            "mood_PA_between_z": pa_between_reference,
            "mood_PA_between_z:n_friends_z_person":
                pa_between_reference * friend_z,
            "gender_female": gender_reference,
            "family_ses_rev_z": ses_reference,
            "nwords_z": word_count_reference,
        }

        design_vector = np.array([
            values[name]
            for name in fixed_effect_names
        ])

        predicted_value = float(
            design_vector @ beta
        )

        prediction_variance = float(
            design_vector
            @ fixed_covariance
            @ design_vector
        )

        prediction_se = np.sqrt(
            max(prediction_variance, 0)
        )

        prediction_rows.append({
            "Positive mood within-person (z)": mood_value,
            "Predicted LLM social connectedness":
                predicted_value,
            "CI lower":
                predicted_value - 1.96 * prediction_se,
            "CI upper":
                predicted_value + 1.96 * prediction_se,
            "Friendship network": friend_label,
        })

pa_predictions = pd.DataFrame(prediction_rows)

# Colors fixed explicitly for consistency
line_colors = {
    "Low: 0 friends": "#C44E52",
    f"Average: {friend_mean:.1f} friends": "#4C72B0",
    f"Higher: {friend_mean + friend_sd:.1f} friends": "#55A868",
}

fig, ax = plt.subplots(figsize=(8, 5.5))

for friend_label in friend_levels_z:

    plot_data = pa_predictions[
        pa_predictions["Friendship network"]
        == friend_label
    ]

    color = line_colors[friend_label]

    ax.plot(
        plot_data["Positive mood within-person (z)"],
        plot_data["Predicted LLM social connectedness"],
        color=color,
        linewidth=2.3,
        label=friend_label,
    )

    ax.fill_between(
        plot_data["Positive mood within-person (z)"],
        plot_data["CI lower"],
        plot_data["CI upper"],
        color=color,
        alpha=0.13,
    )

ax.set_xlabel(
    "Positive mood relative to the participant's usual level (z)"
)

ax.set_ylabel(
    "Predicted LLM social connectedness"
)

ax.set_title(
    "Positive mood × friendship-network size"
)

ax.legend(
    title="Number of nominated friends",
    frameon=False,
)

ax.grid(
    alpha=0.20
)

# Keep the outcome axis consistent with the observed 0–4 scale
ax.set_ylim(
    0,
    4,
)

plt.tight_layout()

pa_figure_path = (
    FIGURES
    / "Figure_PA_within_friends_LLM_common429.png"
)

plt.savefig(
    pa_figure_path,
    dpi=300,
    bbox_inches="tight",
)

plt.show()

print("Saved:", pa_figure_path)
print(
    "Interaction estimate:",
    round(
        pa_llm_model.params[
            "mood_PA_within_z:n_friends_z_person"
        ],
        4,
    ),
)

print(
    "Interaction p-value:",
    round(
        pa_llm_model.pvalues[
            "mood_PA_within_z:n_friends_z_person"
        ],
        4,
    ),
)

In [ ]:
# ============================================================
# FINAL FIGURE 2 — NEGATIVE MOOD × FRIENDS
# Outcome: LLM social connectedness
# ============================================================

# Exact final audited model
na_llm_model = family2_models["NA_social_conn_llm"]

# Use central 90% of observed within-person negative mood
na_grid = np.linspace(
    family2_data["mood_NA_within_z"].quantile(0.05),
    family2_data["mood_NA_within_z"].quantile(0.95),
    100,
)

# Hold other variables at reference values
na_between_reference = 0
ses_reference = 0
word_count_reference = 0
gender_reference = family2_data["gender_female"].mean()

fixed_effect_names = list(
    na_llm_model.fe_params.index
)

beta = na_llm_model.fe_params.loc[
    fixed_effect_names
].to_numpy()

fixed_covariance = na_llm_model.cov_params().loc[
    fixed_effect_names,
    fixed_effect_names,
].to_numpy()

prediction_rows = []

for friend_label, friend_z in friend_levels_z.items():

    for mood_value in na_grid:

        values = {
            "Intercept": 1.0,
            "mood_NA_within_z": mood_value,
            "n_friends_z_person": friend_z,
            "mood_NA_within_z:n_friends_z_person":
                mood_value * friend_z,
            "mood_NA_between_z": na_between_reference,
            "mood_NA_between_z:n_friends_z_person":
                na_between_reference * friend_z,
            "gender_female": gender_reference,
            "family_ses_rev_z": ses_reference,
            "nwords_z": word_count_reference,
        }

        design_vector = np.array([
            values[name]
            for name in fixed_effect_names
        ])

        predicted_value = float(
            design_vector @ beta
        )

        prediction_variance = float(
            design_vector
            @ fixed_covariance
            @ design_vector
        )

        prediction_se = np.sqrt(
            max(prediction_variance, 0)
        )

        prediction_rows.append({
            "Negative mood within-person (z)": mood_value,
            "Predicted LLM social connectedness":
                predicted_value,
            "CI lower":
                predicted_value - 1.96 * prediction_se,
            "CI upper":
                predicted_value + 1.96 * prediction_se,
            "Friendship network": friend_label,
        })

na_predictions = pd.DataFrame(prediction_rows)

fig, ax = plt.subplots(figsize=(8, 5.5))

for friend_label in friend_levels_z:

    plot_data = na_predictions[
        na_predictions["Friendship network"]
        == friend_label
    ]

    color = line_colors[friend_label]

    ax.plot(
        plot_data["Negative mood within-person (z)"],
        plot_data["Predicted LLM social connectedness"],
        color=color,
        linewidth=2.3,
        label=friend_label,
    )

    ax.fill_between(
        plot_data["Negative mood within-person (z)"],
        plot_data["CI lower"],
        plot_data["CI upper"],
        color=color,
        alpha=0.13,
    )

ax.set_xlabel(
    "Negative mood relative to the participant's usual level (z)"
)

ax.set_ylabel(
    "Predicted LLM social connectedness"
)

ax.set_title(
    "Negative mood × friendship-network size"
)

ax.legend(
    title="Number of nominated friends",
    frameon=False,
)

ax.grid(alpha=0.20)
ax.set_ylim(0, 4)

plt.tight_layout()

na_figure_path = (
    FIGURES
    / "Figure_NA_within_friends_LLM_common429.png"
)

plt.savefig(
    na_figure_path,
    dpi=300,
    bbox_inches="tight",
)

plt.show()

print("Saved:", na_figure_path)

print(
    "Interaction estimate:",
    round(
        na_llm_model.params[
            "mood_NA_within_z:n_friends_z_person"
        ],
        4,
    ),
)

print(
    "Interaction p-value:",
    round(
        na_llm_model.pvalues[
            "mood_NA_within_z:n_friends_z_person"
        ],
        4,
    ),
)

In [ ]:
# ============================================================
# FINAL FIGURES 3 — SAVE INTERACTION PREDICTION DATA
# ============================================================

pa_prediction_path = (
    FIGURES
    / "Figure_PA_within_friends_LLM_prediction_data.csv"
)

na_prediction_path = (
    FIGURES
    / "Figure_NA_within_friends_LLM_prediction_data.csv"
)

pa_predictions.to_csv(
    pa_prediction_path,
    index=False,
)

na_predictions.to_csv(
    na_prediction_path,
    index=False,
)

print("Saved:")
print(pa_prediction_path)
print(na_prediction_path)

print("\nPositive-mood prediction rows:", len(pa_predictions))
print("Negative-mood prediction rows:", len(na_predictions))

print(
    "\nFriendship levels in positive-mood figure:",
    pa_predictions["Friendship network"].unique().tolist()
)

print(
    "Friendship levels in negative-mood figure:",
    na_predictions["Friendship network"].unique().tolist()
)

## Final Figure Decision — Friendship-Network Moderation

The interaction figures were generated from the final Family 2 mixed-effects models estimated on the common sample of 429 diary entries from 40 participants. Friendship-network size was represented by the participant-standardized number of nominated friends.

The positive-mood figure presents predictions for participants with 0 friends, the participant-level mean of 2.8 friends, and approximately one standard deviation above the mean, corresponding to 5.6 friends. All three lines slope upward, indicating that higher-than-usual positive mood is associated with higher predicted LLM social connectedness. The slope becomes steeper as friendship-network size increases. However, the interaction is only marginally significant in the primary mixed model (b = 0.058, SE = 0.032, p = .076). The figure therefore illustrates a suggestive positive moderation pattern rather than conclusive evidence of moderation.

The negative-mood figure uses the same friendship-network levels. Predicted LLM social connectedness decreases as negative mood rises above the participant's usual level. The lines are approximately parallel and their confidence intervals overlap substantially. This corresponds to the nonsignificant negative mood × friends interaction (b = -0.026, SE = 0.040, p = .513). Friendship-network size therefore does not clearly moderate the association between within-person negative mood and LLM social connectedness.

The vertical differences between lines should not themselves be interpreted as moderation. The moderation test concerns differences in the slopes across friendship-network levels.

In [ ]:
# ============================================================
# FINAL OUTPUT CHECK — VERIFY ALL AUDITED FILES
# ============================================================

final_outputs = {
    # Audited data
    "Audited analytical checkpoint":
        ROOT / "analysis_ready_audited_v2.parquet",

    # Descriptive tables
    "Diary-level descriptives":
        TABLES / "Table_1a_diary_level_descriptives.csv",

    "Participant-level descriptives":
        TABLES / "Table_1b_participant_level_descriptives.csv",

    "Gender distribution":
        TABLES / "Table_1c_gender_distribution.csv",

    # Correlations
    "Expanded correlation table":
        TABLES / "Table_2_expanded_correlations_common429.csv",

    # Family 1
    "Family 1 thesis table":
        TABLES / "Table_3_Family1_mood_models_common429.csv",

    "Family 1 complete mixed-model results":
        TABLES / "Family1_mixed_models_common429.csv",

    "Family 1 estimator robustness":
        TABLES / "Family1_mixed_vs_clustered_robustness.csv",

    # Family 2
    "Family 2 negative-mood table":
        TABLES / "Table_4a_Family2_NA_friends_common429.csv",

    "Family 2 positive-mood table":
        TABLES / "Table_4b_Family2_PA_friends_common429.csv",

    "Family 2 complete mixed-model results":
        TABLES / "Family2_mixed_models_common429.csv",

    "Family 2 estimator robustness":
        TABLES / "Family2_mixed_vs_clustered_robustness.csv",

    # Family 3
    "Family 3 primary separate-model table":
        TABLES / "Table_5_Family3_separate_PHQ_GAD_common429.csv",

    "Family 3 complete separate-model results":
        TABLES / "Family3_separate_models_full_common429.csv",

    "Family 3 separate-model robustness":
        TABLES / "Family3_separate_models_mixed_vs_clustered.csv",

    "Family 3 joint-model sensitivity results":
        TABLES / "Family3_joint_PHQ_GAD_common429.csv",

    # Figures
    "Positive-mood interaction figure":
        FIGURES / "Figure_PA_within_friends_LLM_common429.png",

    "Negative-mood interaction figure":
        FIGURES / "Figure_NA_within_friends_LLM_common429.png",

    "Positive-mood prediction data":
        FIGURES / "Figure_PA_within_friends_LLM_prediction_data.csv",

    "Negative-mood prediction data":
        FIGURES / "Figure_NA_within_friends_LLM_prediction_data.csv",
}

final_output_check = pd.DataFrame([
    {
        "output": label,
        "exists": path.exists(),
        "path": str(path),
    }
    for label, path in final_outputs.items()
])

display(final_output_check)

missing_final_outputs = final_output_check.loc[
    ~final_output_check["exists"],
    "output",
].tolist()

print("\nExpected outputs:", len(final_output_check))
print("Files available:", int(final_output_check["exists"].sum()))
print("Missing outputs:", missing_final_outputs)

assert not missing_final_outputs, (
    f"Final outputs are missing: {missing_final_outputs}"
)

print("\nFinal output integrity check passed.")

In [ ]:
# ============================================================
# CREATE FINAL THESIS ANALYSIS PACKAGE
# ============================================================

from pathlib import Path
import shutil

PACKAGE = ROOT / "FINAL_THESIS_ANALYSIS"

package_folders = {
    "readme": PACKAGE / "00_README",
    "data": PACKAGE / "01_DATA",
    "descriptives": PACKAGE / "02_DESCRIPTIVES",
    "correlations": PACKAGE / "03_CORRELATIONS",

    "family1_main": PACKAGE / "04_FAMILY1_MOOD" / "MAIN_TABLE",
    "family1_full": PACKAGE / "04_FAMILY1_MOOD" / "FULL_RESULTS",
    "family1_robustness": PACKAGE / "04_FAMILY1_MOOD" / "ROBUSTNESS",

    "family2_main": PACKAGE / "05_FAMILY2_MODERATION" / "MAIN_TABLES",
    "family2_full": PACKAGE / "05_FAMILY2_MODERATION" / "FULL_RESULTS",
    "family2_robustness": PACKAGE / "05_FAMILY2_MODERATION" / "ROBUSTNESS",
    "family2_figures": PACKAGE / "05_FAMILY2_MODERATION" / "FIGURES",

    "family3_main": PACKAGE / "06_FAMILY3_MENTAL_HEALTH" / "MAIN_TABLE",
    "family3_full": PACKAGE / "06_FAMILY3_MENTAL_HEALTH" / "FULL_RESULTS",
    "family3_robustness": PACKAGE / "06_FAMILY3_MENTAL_HEALTH" / "ROBUSTNESS",

    "validation": PACKAGE / "07_VALIDATION",
    "supplementary": PACKAGE / "08_SUPPLEMENTARY",
}

for folder in package_folders.values():
    folder.mkdir(parents=True, exist_ok=True)

files_to_copy = {
    # Audited data
    ROOT / "analysis_ready_audited_v2.parquet":
        package_folders["data"],

    # Descriptives
    TABLES / "Table_1a_diary_level_descriptives.csv":
        package_folders["descriptives"],

    TABLES / "Table_1b_participant_level_descriptives.csv":
        package_folders["descriptives"],

    TABLES / "Table_1c_gender_distribution.csv":
        package_folders["descriptives"],

    # Correlations
    TABLES / "Table_2_expanded_correlations_common429.csv":
        package_folders["correlations"],

    # Family 1
    TABLES / "Table_3_Family1_mood_models_common429.csv":
        package_folders["family1_main"],

    TABLES / "Family1_mixed_models_common429.csv":
        package_folders["family1_full"],

    TABLES / "Family1_mixed_vs_clustered_robustness.csv":
        package_folders["family1_robustness"],

    # Family 2
    TABLES / "Table_4a_Family2_NA_friends_common429.csv":
        package_folders["family2_main"],

    TABLES / "Table_4b_Family2_PA_friends_common429.csv":
        package_folders["family2_main"],

    TABLES / "Family2_mixed_models_common429.csv":
        package_folders["family2_full"],

    TABLES / "Family2_mixed_vs_clustered_robustness.csv":
        package_folders["family2_robustness"],

    FIGURES / "Figure_PA_within_friends_LLM_common429.png":
        package_folders["family2_figures"],

    FIGURES / "Figure_NA_within_friends_LLM_common429.png":
        package_folders["family2_figures"],

    FIGURES / "Figure_PA_within_friends_LLM_prediction_data.csv":
        package_folders["family2_figures"],

    FIGURES / "Figure_NA_within_friends_LLM_prediction_data.csv":
        package_folders["family2_figures"],

    # Family 3
    TABLES / "Table_5_Family3_separate_PHQ_GAD_common429.csv":
        package_folders["family3_main"],

    TABLES / "Family3_separate_models_full_common429.csv":
        package_folders["family3_full"],

    TABLES / "Family3_joint_PHQ_GAD_common429.csv":
        package_folders["family3_full"],

    TABLES / "Family3_separate_models_mixed_vs_clustered.csv":
        package_folders["family3_robustness"],

    # Validation
    OUTPUTS / "validation_all106_HUMAN_LLM_JOINED.csv":
        package_folders["validation"],
}

copy_log = []

for source, destination_folder in files_to_copy.items():

    destination = destination_folder / source.name

    if source.exists():
        shutil.copy2(source, destination)

        copy_log.append({
            "file": source.name,
            "status": "copied",
            "destination": str(destination),
        })

    else:
        copy_log.append({
            "file": source.name,
            "status": "SOURCE MISSING",
            "destination": str(destination),
        })

copy_log = pd.DataFrame(copy_log)

display(copy_log)

print("\nFinal package:", PACKAGE)
print("Files copied:", (copy_log["status"] == "copied").sum())
print(
    "Missing sources:",
    copy_log.loc[
        copy_log["status"] != "copied",
        "file",
    ].tolist()
)

# Minimal Restart Section

After restarting VS Code or the computer, run only the three cells in this section.

In [ ]:
# ============================================================
# MINIMAL STARTUP 1 — IMPORTS AND FINAL PACKAGE PATHS
# ============================================================

from pathlib import Path
import warnings

import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

from scipy import stats
from IPython.display import display

warnings.filterwarnings("default")

# Notebook location:
# ...\Thesis Social Perception\Data Sharing
NOTEBOOK_ROOT = Path.cwd()

# Data location:
# ...\Thesis Social Perception\Data Sharing\Data Sharing
DATA_ROOT = NOTEBOOK_ROOT / "Data Sharing"

# Final organized analysis package
PACKAGE = DATA_ROOT / "FINAL_THESIS_ANALYSIS"

FINAL_DATA = PACKAGE / "01_DATA"
FINAL_DESCRIPTIVES = PACKAGE / "02_DESCRIPTIVES"
FINAL_CORRELATIONS = PACKAGE / "03_CORRELATIONS"
FINAL_FAMILY1 = PACKAGE / "04_FAMILY1_MOOD"
FINAL_FAMILY2 = PACKAGE / "05_FAMILY2_MODERATION"
FINAL_FAMILY3 = PACKAGE / "06_FAMILY3_MENTAL_HEALTH"
FINAL_VALIDATION = PACKAGE / "07_VALIDATION"
FINAL_SUPPLEMENTARY = PACKAGE / "08_SUPPLEMENTARY"

print("Notebook directory:", NOTEBOOK_ROOT)
print("Data root:", DATA_ROOT)
print("Final analysis package:", PACKAGE)
print("Package exists:", PACKAGE.exists())

assert PACKAGE.exists(), (
    f"Final analysis package not found: {PACKAGE}"
)

print("\nFinal package paths ready.")

In [ ]:
# ============================================================
# MINIMAL STARTUP 2 — LOAD AUDITED ANALYTICAL DATA
# ============================================================

analysis = pd.read_parquet(
    FINAL_DATA / "analysis_ready_audited_v2.parquet"
)

analysis_common = analysis[
    analysis["common_language_sample"]
].copy()

print("Full audited panel:", analysis.shape)
print(
    "Full-panel participants:",
    analysis["participantid"].nunique()
)

print("\nCommon comparison sample:", analysis_common.shape)
print(
    "Common-sample participants:",
    analysis_common["participantid"].nunique()
)

print(
    "\nCommon-sample flag available:",
    "common_language_sample" in analysis.columns
)

print(
    "Participant-standardized friend count available:",
    "n_friends_z_person" in analysis.columns
)

# ------------------------------------------------------------
# Integrity checks
# ------------------------------------------------------------

assert len(analysis) == 430, (
    f"Expected 430 rows in the audited panel, found {len(analysis)}."
)

assert len(analysis_common) == 429, (
    f"Expected 429 rows in the common sample, "
    f"found {len(analysis_common)}."
)

assert analysis["participantid"].nunique() == 40, (
    "Expected 40 participants in the audited panel."
)

assert analysis_common["participantid"].nunique() == 40, (
    "Expected 40 participants in the common sample."
)

assert "common_language_sample" in analysis.columns, (
    "The common-language sample flag is missing."
)

assert "n_friends_z_person" in analysis.columns, (
    "The corrected participant-standardized friend variable is missing."
)

print("\nAudited analysis data ready.")

In [ ]:
# ============================================================
# MINIMAL STARTUP 3 — LOAD FINAL THESIS RESULTS
# ============================================================

final_result_paths = {
    # Descriptives
    "Table 1a — Diary-level descriptives":
        FINAL_DESCRIPTIVES
        / "Table_1a_diary_level_descriptives.csv",

    "Table 1b — Participant-level descriptives":
        FINAL_DESCRIPTIVES
        / "Table_1b_participant_level_descriptives.csv",

    "Table 1c — Gender distribution":
        FINAL_DESCRIPTIVES
        / "Table_1c_gender_distribution.csv",

    # Correlations
    "Table 2 — Expanded correlations":
        FINAL_CORRELATIONS
        / "Table_2_expanded_correlations_common429.csv",

    # Family 1
    "Table 3 — Mood models":
        FINAL_FAMILY1
        / "MAIN_TABLE"
        / "Table_3_Family1_mood_models_common429.csv",

    "Family 1 — Full results":
        FINAL_FAMILY1
        / "FULL_RESULTS"
        / "Family1_mixed_models_common429.csv",

    "Family 1 — Robustness":
        FINAL_FAMILY1
        / "ROBUSTNESS"
        / "Family1_mixed_vs_clustered_robustness.csv",

    # Family 2
    "Table 4a — Negative mood moderation":
        FINAL_FAMILY2
        / "MAIN_TABLES"
        / "Table_4a_Family2_NA_friends_common429.csv",

    "Table 4b — Positive mood moderation":
        FINAL_FAMILY2
        / "MAIN_TABLES"
        / "Table_4b_Family2_PA_friends_common429.csv",

    "Family 2 — Full results":
        FINAL_FAMILY2
        / "FULL_RESULTS"
        / "Family2_mixed_models_common429.csv",

    "Family 2 — Robustness":
        FINAL_FAMILY2
        / "ROBUSTNESS"
        / "Family2_mixed_vs_clustered_robustness.csv",

    # Family 3
    "Table 5 — Separate PHQ-9 and GAD-7 models":
        FINAL_FAMILY3
        / "MAIN_TABLE"
        / "Table_5_Family3_separate_PHQ_GAD_common429.csv",

    "Family 3 — Full separate models":
        FINAL_FAMILY3
        / "FULL_RESULTS"
        / "Family3_separate_models_full_common429.csv",

    "Family 3 — Joint-model sensitivity":
        FINAL_FAMILY3
        / "FULL_RESULTS"
        / "Family3_joint_PHQ_GAD_common429.csv",

    "Family 3 — Robustness":
        FINAL_FAMILY3
        / "ROBUSTNESS"
        / "Family3_separate_models_mixed_vs_clustered.csv",
}

loaded_results = {}
result_status = []

for label, path in final_result_paths.items():

    exists = path.exists()

    result_status.append({
        "result": label,
        "exists": exists,
        "path": str(path),
    })

    if exists:
        loaded_results[label] = pd.read_csv(path)

result_status = pd.DataFrame(result_status)

display(result_status)

missing_results = result_status.loc[
    ~result_status["exists"],
    "result",
].tolist()

print("\nExpected result files:", len(result_status))
print("Loaded result files:", len(loaded_results))
print("Missing results:", missing_results)

assert not missing_results, (
    f"Final result files are missing: {missing_results}"
)

print("\nFinal thesis results ready.")